In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [ ]:
# Check GPU and Python environment

import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("⚠️ GPU not available")

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [ ]:
!pip install -q transformers datasets accelerate sentence-transformers scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import transformers
import datasets
import sentence_transformers
import sklearn
import pandas as pd
import numpy as np

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Scikit-learn:", sklearn.__version__)

Transformers: 5.16.1
Datasets: 4.8.5
Sentence Transformers: 5.7.0
Scikit-learn: 1.6.1


In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed fixed at:", SEED)

Random seed fixed at: 42


In [ ]:
PROJECT_NAME = "Context-Aware Multi-Level Intent Drift Detection"

BASE_MODEL = "microsoft/deberta-v3-base"
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

MAX_LENGTH = 256
NUM_LABELS = 2

print("Project:", PROJECT_NAME)
print("Main Transformer:", BASE_MODEL)
print("Semantic Model:", SBERT_MODEL)
print("Maximum sequence length:", MAX_LENGTH)

Project: Context-Aware Multi-Level Intent Drift Detection
Main Transformer: microsoft/deberta-v3-base
Semantic Model: sentence-transformers/all-MiniLM-L6-v2
Maximum sequence length: 256


In [ ]:
import requests
import os
import json

BASE_URL = "https://raw.githubusercontent.com/budzianowski/multiwoz/master/data/MultiWOZ_2.2"

splits = {
    "train": "train/dialogues_001.json",
    "validation": "dev/dialogues_001.json",
    "test": "test/dialogues_001.json"
}

for split, file_path in splits.items():
    url = f"{BASE_URL}/{file_path}"
    response = requests.get(url)

    print(split, "status:", response.status_code)

    if response.status_code == 200:
        data = response.json()
        print(split, "dialogues:", len(data))
    else:
        print("Failed:", url)

train status: 200
train dialogues: 512
validation status: 200
validation dialogues: 512
test status: 200
test dialogues: 512


In [ ]:
import requests
import os
import json

BASE_URL = "https://raw.githubusercontent.com/budzianowski/multiwoz/master/data/MultiWOZ_2.2"

splits = {
    "train": "train",
    "validation": "dev",
    "test": "test"
}

all_data = {}

for split, folder in splits.items():
    all_data[split] = []

    for i in range(1, 30):
        filename = f"dialogues_{i:03d}.json"
        url = f"{BASE_URL}/{folder}/{filename}"

        response = requests.get(url)

        if response.status_code == 200:
            data = response.json()
            all_data[split].extend(data)
            print(f"{split}: {filename} -> {len(data)} dialogues")
        else:
            break

    print(f"\nTotal {split} dialogues:", len(all_data[split]))
    print("-" * 50)

print("\nDATASET DOWNLOAD COMPLETE")

train: dialogues_001.json -> 512 dialogues
train: dialogues_002.json -> 512 dialogues
train: dialogues_003.json -> 512 dialogues
train: dialogues_004.json -> 512 dialogues
train: dialogues_005.json -> 512 dialogues
train: dialogues_006.json -> 512 dialogues
train: dialogues_007.json -> 511 dialogues
train: dialogues_008.json -> 512 dialogues
train: dialogues_009.json -> 512 dialogues
train: dialogues_010.json -> 512 dialogues
train: dialogues_011.json -> 512 dialogues
train: dialogues_012.json -> 512 dialogues
train: dialogues_013.json -> 512 dialogues
train: dialogues_014.json -> 512 dialogues
train: dialogues_015.json -> 512 dialogues
train: dialogues_016.json -> 512 dialogues
train: dialogues_017.json -> 246 dialogues

Total train dialogues: 8437
--------------------------------------------------
validation: dialogues_001.json -> 512 dialogues
validation: dialogues_002.json -> 488 dialogues

Total validation dialogues: 1000
--------------------------------------------------
test: di

In [ ]:
# Inspect one complete dialogue

sample_dialogue = all_data["train"][0]

print("Dialogue ID:", sample_dialogue["dialogue_id"])
print("Number of turns:", len(sample_dialogue["turns"]))
print("\nFirst turn:")
print(json.dumps(sample_dialogue["turns"][0], indent=2))

Dialogue ID: PMUL4398.json
Number of turns: 12

First turn:
{
  "frames": [
    {
      "actions": [],
      "service": "restaurant",
      "slots": [],
      "state": {
        "active_intent": "find_restaurant",
        "requested_slots": [],
        "slot_values": {
          "restaurant-area": [
            "centre"
          ],
          "restaurant-pricerange": [
            "expensive"
          ]
        }
      }
    },
    {
      "actions": [],
      "service": "taxi",
      "slots": [],
      "state": {
        "active_intent": "NONE",
        "requested_slots": [],
        "slot_values": {}
      }
    },
    {
      "actions": [],
      "service": "train",
      "slots": [],
      "state": {
        "active_intent": "NONE",
        "requested_slots": [],
        "slot_values": {}
      }
    },
    {
      "actions": [],
      "service": "bus",
      "slots": [],
      "state": {
        "active_intent": "NONE",
        "requested_slots": [],
        "slot_values": {}
   

In [ ]:
# Extract USER turns with intent and domain information

def extract_user_turns(dialogue):
    user_turns = []

    for turn in dialogue["turns"]:
        if turn["speaker"] != "USER":
            continue

        intents = []
        domains = []

        for frame in turn["frames"]:
            intent = frame["state"]["active_intent"]
            service = frame["service"]

            if intent != "NONE":
                intents.append(intent)

            if intent != "NONE":
                domains.append(service)

        user_turns.append({
            "dialogue_id": dialogue["dialogue_id"],
            "turn_id": int(turn["turn_id"]),
            "utterance": turn["utterance"],
            "intent": intents[0] if intents else "NONE",
            "domain": domains[0] if domains else "NONE"
        })

    return user_turns


# Test on the first dialogue
sample_user_turns = extract_user_turns(all_data["train"][0])

for turn in sample_user_turns:
    print(turn)

{'dialogue_id': 'PMUL4398.json', 'turn_id': 0, 'utterance': 'i need a place to dine in the center thats expensive', 'intent': 'find_restaurant', 'domain': 'restaurant'}
{'dialogue_id': 'PMUL4398.json', 'turn_id': 2, 'utterance': 'Any sort of food would be fine, as long as it is a bit expensive. Could I get the phone number for your recommendation?', 'intent': 'find_restaurant', 'domain': 'restaurant'}
{'dialogue_id': 'PMUL4398.json', 'turn_id': 4, 'utterance': 'Sounds good, could I get that phone number? Also, could you recommend me an expensive hotel?', 'intent': 'find_restaurant', 'domain': 'restaurant'}
{'dialogue_id': 'PMUL4398.json', 'turn_id': 6, 'utterance': 'Yes. Can you book it for me?', 'intent': 'find_hotel', 'domain': 'hotel'}
{'dialogue_id': 'PMUL4398.json', 'turn_id': 8, 'utterance': 'i want to book it for 2 people and 2 nights starting from saturday.', 'intent': 'book_hotel', 'domain': 'hotel'}
{'dialogue_id': 'PMUL4398.json', 'turn_id': 10, 'utterance': 'That is all I n

In [ ]:
# Preserve ALL active intents and domains for each USER turn

def extract_user_turns(dialogue):
    user_turns = []

    for turn in dialogue["turns"]:
        if turn["speaker"] != "USER":
            continue

        intents = []
        domains = []

        for frame in turn["frames"]:
            intent = frame["state"]["active_intent"]
            service = frame["service"]

            if intent != "NONE" and intent not in intents:
                intents.append(intent)

            if intent != "NONE" and service not in domains:
                domains.append(service)

        user_turns.append({
            "dialogue_id": dialogue["dialogue_id"],
            "turn_id": int(turn["turn_id"]),
            "utterance": turn["utterance"],
            "intents": intents,
            "domains": domains
        })

    return user_turns


# Test again
sample_user_turns = extract_user_turns(all_data["train"][0])

for turn in sample_user_turns:
    print(turn)

{'dialogue_id': 'PMUL4398.json', 'turn_id': 0, 'utterance': 'i need a place to dine in the center thats expensive', 'intents': ['find_restaurant', 'find_hotel'], 'domains': ['restaurant', 'hotel']}
{'dialogue_id': 'PMUL4398.json', 'turn_id': 2, 'utterance': 'Any sort of food would be fine, as long as it is a bit expensive. Could I get the phone number for your recommendation?', 'intents': ['find_restaurant', 'find_hotel'], 'domains': ['restaurant', 'hotel']}
{'dialogue_id': 'PMUL4398.json', 'turn_id': 4, 'utterance': 'Sounds good, could I get that phone number? Also, could you recommend me an expensive hotel?', 'intents': ['find_restaurant', 'find_hotel'], 'domains': ['restaurant', 'hotel']}
{'dialogue_id': 'PMUL4398.json', 'turn_id': 6, 'utterance': 'Yes. Can you book it for me?', 'intents': ['find_hotel'], 'domains': ['hotel']}
{'dialogue_id': 'PMUL4398.json', 'turn_id': 8, 'utterance': 'i want to book it for 2 people and 2 nights starting from saturday.', 'intents': ['book_hotel'], 

In [ ]:
import requests
import json

url = "https://raw.githubusercontent.com/budzianowski/multiwoz/master/data/MultiWOZ_2.2/dialog_acts.json"

response = requests.get(url)

print("Status code:", response.status_code)

if response.status_code == 200:
    dialogue_acts = response.json()

    print("Dialogue acts loaded successfully!")
    print("Number of dialogues:", len(dialogue_acts))

    first_id = list(dialogue_acts.keys())[0]
    print("\nExample dialogue:", first_id)
    print(json.dumps(dialogue_acts[first_id], indent=2)[:3000])
else:
    print("Failed to download dialogue acts")

Status code: 200
Dialogue acts loaded successfully!
Number of dialogues: 10437

Example dialogue: SNG01856.json
{
  "0": {
    "dialog_act": {
      "Hotel-Inform": [
        [
          "pricerange",
          "cheap"
        ],
        [
          "type",
          "hotel"
        ]
      ]
    },
    "span_info": [
      [
        "Hotel-Inform",
        "pricerange",
        "cheap",
        43,
        48
      ],
      [
        "Hotel-Inform",
        "type",
        "hotel",
        87,
        92
      ]
    ]
  },
  "1": {
    "dialog_act": {
      "Hotel-Request": [
        [
          "area",
          "?"
        ]
      ]
    },
    "span_info": []
  },
  "2": {
    "dialog_act": {
      "Hotel-Inform": [
        [
          "parking",
          "yes"
        ]
      ]
    },
    "span_info": []
  },
  "3": {
    "dialog_act": {
      "Booking-Inform": [
        [
          "none",
          "none"
        ]
      ],
      "Hotel-Inform": [
        [
          "choice",
 

In [ ]:
import pandas as pd

def get_dialogue_act_info(dialogue_id, turn_id):
    """
    Extract dialogue-act information for a specific turn.
    """
    dialogue_info = dialogue_acts.get(dialogue_id, {})
    turn_info = dialogue_info.get(str(turn_id), {})

    acts = turn_info.get("dialog_act", {})

    domains = []
    actions = []

    for act_name in acts.keys():
        parts = act_name.split("-", 1)

        if len(parts) == 2:
            domain, action = parts
            domains.append(domain.lower())
            actions.append(action.lower())

    return list(set(domains)), list(set(actions))


rows = []

for dialogue in all_data["train"]:
    dialogue_id = dialogue["dialogue_id"]

    for turn in dialogue["turns"]:
        turn_id = int(turn["turn_id"])

        domains, actions = get_dialogue_act_info(
            dialogue_id,
            turn_id
        )

        rows.append({
            "dialogue_id": dialogue_id,
            "turn_id": turn_id,
            "speaker": turn["speaker"],
            "utterance": turn["utterance"],
            "domains": domains,
            "actions": actions
        })

train_turns = pd.DataFrame(rows)

print("Total turns:", len(train_turns))
print("\nColumns:")
print(train_turns.columns.tolist())

print("\nFirst 10 rows:")
display(train_turns.head(10))

Total turns: 113552

Columns:
['dialogue_id', 'turn_id', 'speaker', 'utterance', 'domains', 'actions']

First 10 rows:


,dialogue_id,turn_id,speaker,utterance,domains,actions
0,PMUL4398.json,0,USER,i need a place to dine in the center thats exp...,[restaurant],[inform]
1,PMUL4398.json,1,SYSTEM,I have several options for you; do you prefer ...,[restaurant],"[inform, select]"
2,PMUL4398.json,2,USER,"Any sort of food would be fine, as long as it ...",[restaurant],[request]
3,PMUL4398.json,3,SYSTEM,There is an Afrian place named Bedouin in the ...,[restaurant],[inform]
4,PMUL4398.json,4,USER,"Sounds good, could I get that phone number? Al...","[restaurant, hotel]","[inform, request]"
5,PMUL4398.json,5,SYSTEM,Bedouin's phone is 01223367660. As far as hote...,"[restaurant, hotel]","[recommend, inform]"
6,PMUL4398.json,6,USER,Yes. Can you book it for me?,[hotel],[inform]
7,PMUL4398.json,7,SYSTEM,"Sure, when would you like that reservation?",[booking],[request]
8,PMUL4398.json,8,USER,i want to book it for 2 people and 2 nights st...,[hotel],[inform]
9,PMUL4398.json,9,SYSTEM,Your booking was successful. Your reference nu...,"[general, booking]","[reqmore, book]"


In [ ]:
# Build USER-to-USER transition pairs

user_turns = train_turns[
    train_turns["speaker"] == "USER"
].copy()

transition_rows = []

for dialogue_id, group in user_turns.groupby("dialogue_id"):
    group = group.sort_values("turn_id").reset_index(drop=True)

    for i in range(len(group) - 1):
        previous = group.iloc[i]
        current = group.iloc[i + 1]

        transition_rows.append({
            "dialogue_id": dialogue_id,

            "previous_turn_id": previous["turn_id"],
            "current_turn_id": current["turn_id"],

            "previous_utterance": previous["utterance"],
            "current_utterance": current["utterance"],

            "previous_domains": previous["domains"],
            "current_domains": current["domains"],

            "previous_actions": previous["actions"],
            "current_actions": current["actions"]
        })

train_transitions = pd.DataFrame(transition_rows)

print("Total USER-to-USER transitions:", len(train_transitions))

print("\nColumns:")
print(train_transitions.columns.tolist())

print("\nFirst 5 transitions:")
display(train_transitions.head())

Total USER-to-USER transitions: 48339

Columns:
['dialogue_id', 'previous_turn_id', 'current_turn_id', 'previous_utterance', 'current_utterance', 'previous_domains', 'current_domains', 'previous_actions', 'current_actions']

First 5 transitions:


,dialogue_id,previous_turn_id,current_turn_id,previous_utterance,current_utterance,previous_domains,current_domains,previous_actions,current_actions
0,MUL0001.json,0,2,I'd really like to take my client out to a nic...,I am looking for an expensive indian restauran...,[restaurant],[restaurant],[inform],[inform]
1,MUL0001.json,2,4,I am looking for an expensive indian restauran...,"Sure thing, please book for 6 people at 19:30 ...",[restaurant],[restaurant],[inform],[inform]
2,MUL0001.json,4,6,"Sure thing, please book for 6 people at 19:30 ...",Okay great! Thank you so much. Could you also ...,[restaurant],[hotel],[inform],[inform]
3,MUL0001.json,6,8,Okay great! Thank you so much. Could you also ...,That sounds great. Please book that now.,[hotel],[hotel],[inform],[inform]
4,MUL0001.json,8,10,That sounds great. Please book that now.,I have 6 people in my group.,[hotel],[hotel],[inform],[inform]


In [ ]:
# Create initial multi-level intent drift labels

def determine_drift(row):
    prev_domains = set(row["previous_domains"])
    curr_domains = set(row["current_domains"])

    prev_actions = set(row["previous_actions"])
    curr_actions = set(row["current_actions"])

    # Handle empty annotations
    if not curr_domains:
        return "NO_DRIFT"

    # Multiple domains in current turn
    if len(curr_domains) > 1:
        return "MULTI_GOAL"

    # Domain changed
    if prev_domains and curr_domains and prev_domains != curr_domains:
        return "DOMAIN_CHANGE"

    # Same domain but action/goal changed
    if prev_actions and curr_actions and prev_actions != curr_actions:
        return "GOAL_CHANGE"

    # Same domain and same action
    return "NO_DRIFT"


train_transitions["drift_label"] = train_transitions.apply(
    determine_drift,
    axis=1
)

print("Drift label distribution:")
print(train_transitions["drift_label"].value_counts())

print("\nSample labeled transitions:")
display(
    train_transitions[
        [
            "previous_utterance",
            "current_utterance",
            "previous_domains",
            "current_domains",
            "previous_actions",
            "current_actions",
            "drift_label"
        ]
    ].head(10)
)



Drift label distribution:
drift_label
NO_DRIFT         21775
DOMAIN_CHANGE    15700
GOAL_CHANGE       9399
MULTI_GOAL        1465
Name: count, dtype: int64

Sample labeled transitions:


,previous_utterance,current_utterance,previous_domains,current_domains,previous_actions,current_actions,drift_label
0,I'd really like to take my client out to a nic...,I am looking for an expensive indian restauran...,[restaurant],[restaurant],[inform],[inform],NO_DRIFT
1,I am looking for an expensive indian restauran...,"Sure thing, please book for 6 people at 19:30 ...",[restaurant],[restaurant],[inform],[inform],NO_DRIFT
2,"Sure thing, please book for 6 people at 19:30 ...",Okay great! Thank you so much. Could you also ...,[restaurant],[hotel],[inform],[inform],DOMAIN_CHANGE
3,Okay great! Thank you so much. Could you also ...,That sounds great. Please book that now.,[hotel],[hotel],[inform],[inform],NO_DRIFT
4,That sounds great. Please book that now.,I have 6 people in my group.,[hotel],[hotel],[inform],[inform],NO_DRIFT
5,I have 6 people in my group.,"2 nights, starting the same day as the reserva...",[hotel],[hotel],[inform],[inform],NO_DRIFT
6,"2 nights, starting the same day as the reserva...",Yes please. Is there something else available ...,[hotel],[],[inform],[],NO_DRIFT
7,Yes please. Is there something else available ...,Can we try for 1 night instead of 2? I'll need...,[],[hotel],[],[inform],NO_DRIFT
8,Can we try for 1 night instead of 2? I'll need...,Thank you so much!,[hotel],[general],[inform],[thank],DOMAIN_CHANGE
9,Looking for a place to eat in the city center.,I'd like to have some Chinese food.,[restaurant],[restaurant],[inform],"[inform, request]",GOAL_CHANGE


In [ ]:
# Inspect the most common domain and action transitions
# Convert list-based domain/action fields to strings
# only for frequency analysis.

analysis_df = train_transitions.copy()

analysis_df["previous_domains_str"] = analysis_df["previous_domains"].apply(
    lambda x: ", ".join(sorted(x)) if x else "NONE"
)

analysis_df["current_domains_str"] = analysis_df["current_domains"].apply(
    lambda x: ", ".join(sorted(x)) if x else "NONE"
)

analysis_df["previous_actions_str"] = analysis_df["previous_actions"].apply(
    lambda x: ", ".join(sorted(x)) if x else "NONE"
)

analysis_df["current_actions_str"] = analysis_df["current_actions"].apply(
    lambda x: ", ".join(sorted(x)) if x else "NONE"
)


print("TOP DOMAIN TRANSITIONS")
print("=" * 60)

domain_transitions = (
    analysis_df
    .groupby(
        ["previous_domains_str", "current_domains_str"]
    )
    .size()
    .sort_values(ascending=False)
    .head(30)
)

print(domain_transitions)


print("\n\nTOP ACTION TRANSITIONS")
print("=" * 60)

action_transitions = (
    analysis_df
    .groupby(
        ["previous_actions_str", "current_actions_str"]
    )
    .size()
    .sort_values(ascending=False)
    .head(30)
)

print(action_transitions)

TOP DOMAIN TRANSITIONS
previous_domains_str  current_domains_str
hotel                 hotel                  8299
restaurant            restaurant             7674
train                 train                  6667
attraction            attraction             3628
restaurant            general                2083
hotel                 general                1590
train                 general                1580
general               general                1574
taxi                  general                1036
                      taxi                   1017
attraction            general                 971
train                 hotel                   486
attraction            hotel                   460
restaurant            train                   454
attraction            train                   447
hotel                 restaurant              432
                      train                   427
restaurant            hotel                   398
train                 attraction   

In [ ]:
# Create research-oriented functional domains
# by separating task-related domains from conversational "general".

CONVERSATIONAL_DOMAINS = {"general"}

def clean_domains(domains):
    """
    Remove conversational domains such as 'general'.
    """
    if not domains:
        return set()

    return {
        d for d in domains
        if d not in CONVERSATIONAL_DOMAINS
    }


def create_research_label(row):
    prev_domains = clean_domains(row["previous_domains"])
    curr_domains = clean_domains(row["current_domains"])

    prev_actions = set(row["previous_actions"])
    curr_actions = set(row["current_actions"])

    # Current utterance contains multiple functional domains
    if len(curr_domains) > 1:
        return "MULTI_GOAL"

    # No functional domain in current turn
    # e.g., "thanks", "goodbye"
    if not curr_domains:
        return "NO_DRIFT"

    # Previous turn had no functional domain
    # and current turn starts a task
    if not prev_domains:
        return "NO_DRIFT"

    # Functional domain changed
    if prev_domains != curr_domains:
        return "DOMAIN_CHANGE"

    # Same functional domain but dialogue function changed
    if prev_actions != curr_actions:
        return "GOAL_CHANGE"

    # Same domain and same dialogue function
    return "NO_DRIFT"


train_transitions["research_label"] = train_transitions.apply(
    create_research_label,
    axis=1
)

print("RESEARCH LABEL DISTRIBUTION")
print("=" * 50)

print(
    train_transitions["research_label"]
    .value_counts()
)

print("\nPercentage distribution:")
print(
    (train_transitions["research_label"]
     .value_counts(normalize=True) * 100)
    .round(2)
)

RESEARCH LABEL DISTRIBUTION
research_label
NO_DRIFT         31204
GOAL_CHANGE       8849
DOMAIN_CHANGE     6821
MULTI_GOAL        1465
Name: count, dtype: int64

Percentage distribution:
research_label
NO_DRIFT         64.55
GOAL_CHANGE      18.31
DOMAIN_CHANGE    14.11
MULTI_GOAL        3.03
Name: proportion, dtype: float64


In [ ]:
# Inspect representative examples from each research label

labels_to_check = [
    "NO_DRIFT",
    "GOAL_CHANGE",
    "DOMAIN_CHANGE",
    "MULTI_GOAL"
]

for label in labels_to_check:
    print("\n" + "=" * 80)
    print("LABEL:", label)
    print("=" * 80)

    examples = train_transitions[
        train_transitions["research_label"] == label
    ].sample(
        n=min(5, len(
            train_transitions[
                train_transitions["research_label"] == label
            ]
        )),
        random_state=42
    )

    for _, row in examples.iterrows():
        print("\nPrevious:")
        print(row["previous_utterance"])

        print("Current:")
        print(row["current_utterance"])

        print(
            "Domains:",
            row["previous_domains"],
            "→",
            row["current_domains"]
        )

        print(
            "Actions:",
            row["previous_actions"],
            "→",
            row["current_actions"]
        )


LABEL: NO_DRIFT

Previous:
No, I do not.
Current:
Sure, book it for 3 people please.
Domains: ['train'] → ['train']
Actions: ['inform'] → ['inform']

Previous:
Well then yes, 3 stars would be OK.
Current:
Area doesn't matter. You can recommend the one you like best, and then please book it for 5 people and 2 nights starting sunday.
Domains: ['hotel'] → ['hotel']
Actions: ['inform'] → ['inform']

Previous:
I would like to depart from kings lynn.
Current:
I'd like to arrive no later than 9:00 in the morning.
Domains: ['train'] → ['general']
Actions: ['inform'] → ['thank']

Previous:
I don't have preference on type of place, but it should have 4 stars.
Current:
No, I'm not worried about those things, just as long as it has 4 star, free parking, and doesn't need to include Internet.
Domains: ['hotel'] → ['hotel']
Actions: ['inform'] → ['inform']

Previous:
I am going to allenbell
Current:
Thats perfect that you for your help
Domains: ['taxi'] → ['taxi']
Actions: ['inform'] → ['inform']

L

In [ ]:
# Inspect frame-level active intents for USER turns.
# This helps us distinguish a real task/intent change
# from a simple inform/request dialogue-act change.

def get_user_intents_from_frames(dialogue, turn_id):
    for turn in dialogue["turns"]:
        if int(turn["turn_id"]) == turn_id and turn["speaker"] == "USER":

            intents = []
            domains = []

            for frame in turn["frames"]:
                intent = frame["state"]["active_intent"]
                service = frame["service"]

                if intent != "NONE" and intent not in intents:
                    intents.append(intent)

                if intent != "NONE" and service not in domains:
                    domains.append(service)

            return intents, domains

    return [], []


# Inspect a few USER-to-USER transitions
sample = train_transitions.sample(10, random_state=42)

for _, row in sample.iterrows():

    dialogue = next(
        d for d in all_data["train"]
        if d["dialogue_id"] == row["dialogue_id"]
    )

    prev_intents, prev_domains = get_user_intents_from_frames(
        dialogue,
        row["previous_turn_id"]
    )

    curr_intents, curr_domains = get_user_intents_from_frames(
        dialogue,
        row["current_turn_id"]
    )

    print("=" * 80)

    print("PREVIOUS:")
    print(row["previous_utterance"])
    print("Intent:", prev_intents)
    print("Domain:", prev_domains)

    print("\nCURRENT:")
    print(row["current_utterance"])
    print("Intent:", curr_intents)
    print("Domain:", curr_domains)

    print()

PREVIOUS:
Try 2 nights, then.
Intent: ['book_hotel']
Domain: ['hotel']

CURRENT:
Yes, I am also looking for a train that leaves after 14:15 and goes to bishops stortford.
Intent: ['find_train']
Domain: ['train']

PREVIOUS:
That sounds great. I would like to book that for 3 people and 2 nights starting from Sunday.
Intent: ['book_hotel']
Domain: ['hotel']

CURRENT:
No, that is all I need today. Thank you.
Intent: []
Domain: []

PREVIOUS:
I would like Italian please.
Intent: ['find_restaurant']
Domain: ['restaurant']

CURRENT:
It doesn't matter the part of town.
Intent: ['find_restaurant']
Domain: ['restaurant']

PREVIOUS:
Somewhere in the west please.
Intent: ['find_hotel']
Domain: ['hotel']

CURRENT:
I don't have a price range in mind.
Intent: ['find_hotel']
Domain: ['hotel']

PREVIOUS:
I'd like the attraction type, phone number, and entrance fee, please.
Intent: ['find_attraction']
Domain: ['attraction']

CURRENT:
Alright, that's all. Thank you.
Intent: []
Domain: []

PREVIOUS:
I will

In [ ]:
# Create intent-aware USER-to-USER transition dataset

def get_turn_intent_info(dialogue, turn_id):
    """
    Extract active intents and domains from a specific USER turn.
    """
    for turn in dialogue["turns"]:
        if (
            int(turn["turn_id"]) == turn_id
            and turn["speaker"] == "USER"
        ):
            intents = []
            domains = []

            for frame in turn["frames"]:
                intent = frame["state"]["active_intent"]
                service = frame["service"]

                if intent != "NONE" and intent not in intents:
                    intents.append(intent)

                if intent != "NONE" and service not in domains:
                    domains.append(service)

            return intents, domains

    return [], []


# Cache dialogues for faster lookup
train_dialogue_dict = {
    d["dialogue_id"]: d
    for d in all_data["train"]
}


intent_transition_rows = []

for _, row in train_transitions.iterrows():

    dialogue = train_dialogue_dict[row["dialogue_id"]]

    prev_intents, prev_domains = get_turn_intent_info(
        dialogue,
        row["previous_turn_id"]
    )

    curr_intents, curr_domains = get_turn_intent_info(
        dialogue,
        row["current_turn_id"]
    )

    # Determine research label

    if len(curr_intents) > 1 or len(curr_domains) > 1:
        label = "MULTI_GOAL"

    elif len(curr_intents) == 0:
        label = "NO_DRIFT"

    elif len(prev_intents) == 0:
        label = "NO_DRIFT"

    elif set(prev_domains) != set(curr_domains):
        label = "DOMAIN_CHANGE"

    elif set(prev_intents) != set(curr_intents):
        label = "GOAL_CHANGE"

    else:
        label = "NO_DRIFT"

    intent_transition_rows.append({
        "dialogue_id": row["dialogue_id"],
        "previous_turn_id": row["previous_turn_id"],
        "current_turn_id": row["current_turn_id"],

        "previous_utterance": row["previous_utterance"],
        "current_utterance": row["current_utterance"],

        "previous_intents": prev_intents,
        "current_intents": curr_intents,

        "previous_domains": prev_domains,
        "current_domains": curr_domains,

        "drift_label": label
    })


intent_transitions = pd.DataFrame(intent_transition_rows)

print("Total intent-aware transitions:",
      len(intent_transitions))

print("\nLabel distribution:")
print(
    intent_transitions["drift_label"]
    .value_counts()
)

print("\nPercentage:")
print(
    (
        intent_transitions["drift_label"]
        .value_counts(normalize=True) * 100
    ).round(2)
)

Total intent-aware transitions: 48339

Label distribution:
drift_label
NO_DRIFT         30057
DOMAIN_CHANGE     8698
MULTI_GOAL        5691
GOAL_CHANGE       3893
Name: count, dtype: int64

Percentage:
drift_label
NO_DRIFT         62.18
DOMAIN_CHANGE    17.99
MULTI_GOAL       11.77
GOAL_CHANGE       8.05
Name: proportion, dtype: float64


In [ ]:
# Final validation of the intent-aware labels

for label in [
    "NO_DRIFT",
    "GOAL_CHANGE",
    "DOMAIN_CHANGE",
    "MULTI_GOAL"
]:

    print("\n" + "=" * 90)
    print("LABEL:", label)
    print("=" * 90)

    examples = intent_transitions[
        intent_transitions["drift_label"] == label
    ].sample(
        n=5,
        random_state=42
    )

    for _, row in examples.iterrows():

        print("\nPREVIOUS:")
        print(row["previous_utterance"])
        print("Intent:", row["previous_intents"])
        print("Domain:", row["previous_domains"])

        print("\nCURRENT:")
        print(row["current_utterance"])
        print("Intent:", row["current_intents"])
        print("Domain:", row["current_domains"])

        print("\nLABEL:", row["drift_label"])
        print("-" * 90)


LABEL: NO_DRIFT

PREVIOUS:
Ok, how about an Italian restaurant?
Intent: ['find_restaurant']
Domain: ['restaurant']

CURRENT:
What is the phone number for prezzo?
Intent: ['find_restaurant']
Domain: ['restaurant']

LABEL: NO_DRIFT
------------------------------------------------------------------------------------------

PREVIOUS:
I need a place to stay with free wifi.
Intent: ['find_hotel']
Domain: ['hotel']

CURRENT:
Location doesn't really matter as long as there's free parking available.
Intent: ['find_hotel']
Domain: ['hotel']

LABEL: NO_DRIFT
------------------------------------------------------------------------------------------

PREVIOUS:
I would like to leave after 9:45.
Intent: ['find_train']
Domain: ['train']

CURRENT:
I just need the travel time please.
Intent: ['find_train']
Domain: ['train']

LABEL: NO_DRIFT
------------------------------------------------------------------------------------------

PREVIOUS:
Thanks so much! Can you help me with train tickets too?
Intent

In [ ]:
# Refine the intent-drift labeling logic
# Important: a subset of previously active goals is NOT a domain change.

def refined_drift_label(row):

    prev_intents = set(row["previous_intents"])
    curr_intents = set(row["current_intents"])

    prev_domains = set(row["previous_domains"])
    curr_domains = set(row["current_domains"])

    # --------------------------------------------------
    # 1. No current task intent
    #    Example: "Thanks, that's all."
    # --------------------------------------------------
    if not curr_intents:
        return "NO_DRIFT"

    # --------------------------------------------------
    # 2. Current turn contains multiple domains/goals
    # --------------------------------------------------
    if len(curr_domains) > 1 or len(curr_intents) > 1:
        return "MULTI_GOAL"

    # --------------------------------------------------
    # 3. No previous task
    # --------------------------------------------------
    if not prev_intents:
        return "NO_DRIFT"

    # --------------------------------------------------
    # 4. Current goal is already contained in previous goals
    #    Example:
    #    previous = {find_restaurant, find_train}
    #    current  = {find_train}
    #
    #    This is continuation, NOT domain change.
    # --------------------------------------------------
    if curr_intents.issubset(prev_intents):
        return "NO_DRIFT"

    # --------------------------------------------------
    # 5. Same domain but different task intent
    #    Example:
    #    find_restaurant -> book_restaurant
    # --------------------------------------------------
    if prev_domains == curr_domains:
        return "GOAL_CHANGE"

    # --------------------------------------------------
    # 6. Current domain is genuinely different
    # --------------------------------------------------
    if not curr_domains.intersection(prev_domains):
        return "DOMAIN_CHANGE"

    # --------------------------------------------------
    # 7. Otherwise, treat as continuation
    # --------------------------------------------------
    return "NO_DRIFT"


# Apply refined labels
intent_transitions["drift_label"] = intent_transitions.apply(
    refined_drift_label,
    axis=1
)

print("Refined label distribution:")
print(intent_transitions["drift_label"].value_counts())

print("\nRefined percentages:")
print(
    (
        intent_transitions["drift_label"]
        .value_counts(normalize=True) * 100
    ).round(2)
)

Refined label distribution:
drift_label
NO_DRIFT         34255
MULTI_GOAL        5691
DOMAIN_CHANGE     4500
GOAL_CHANGE       3893
Name: count, dtype: int64

Refined percentages:
drift_label
NO_DRIFT         70.86
MULTI_GOAL       11.77
DOMAIN_CHANGE     9.31
GOAL_CHANGE       8.05
Name: proportion, dtype: float64


In [ ]:
# Validate the refined DOMAIN_CHANGE and MULTI_GOAL labels

for label in ["DOMAIN_CHANGE", "MULTI_GOAL"]:

    print("\n" + "=" * 100)
    print("REFINED LABEL:", label)
    print("=" * 100)

    examples = intent_transitions[
        intent_transitions["drift_label"] == label
    ].sample(
        n=10,
        random_state=42
    )

    for _, row in examples.iterrows():

        print("\nPREVIOUS:")
        print(row["previous_utterance"])
        print("Intents:", row["previous_intents"])
        print("Domains:", row["previous_domains"])

        print("\nCURRENT:")
        print(row["current_utterance"])
        print("Intents:", row["current_intents"])
        print("Domains:", row["current_domains"])

        print("\nLABEL:", row["drift_label"])
        print("-" * 100)


REFINED LABEL: DOMAIN_CHANGE

PREVIOUS:
Yes, please book a ticket.
Intents: ['find_train']
Domains: ['train']

CURRENT:
No let's not book that. I want to hear about the attraction with architecture.
Intents: ['find_attraction']
Domains: ['attraction']

LABEL: DOMAIN_CHANGE
----------------------------------------------------------------------------------------------------

PREVIOUS:
Yes please. Can you also provide me with the postcode?
Intents: ['find_hotel']
Domains: ['hotel']

CURRENT:
Could you assist me in finding a restaurant? We are going all out so would like an expensive one near the hotel.
Intents: ['find_restaurant']
Domains: ['restaurant']

LABEL: DOMAIN_CHANGE
----------------------------------------------------------------------------------------------------

PREVIOUS:
5 nights please. And I would need the reference number also.
Intents: ['book_hotel']
Domains: ['hotel']

CURRENT:
Thank you! I am also looking for a place to go in town. I'm thinking a college in the centr

In [ ]:
# Research-grade intent filtering:
# Keep active intents whose service/domain is actually expressed
# by the current USER turn's dialogue acts.

def get_current_act_domains(dialogue, turn_id):

    for turn in dialogue["turns"]:

        if (
            int(turn["turn_id"]) == turn_id
            and turn["speaker"] == "USER"
        ):

            domains = set()

            dialogue_info = dialogue_acts.get(
                dialogue["dialogue_id"], {}
            )

            turn_act_info = dialogue_info.get(
                str(turn_id), {}
            )

            acts = turn_act_info.get(
                "dialog_act", {}
            )

            for act_name in acts.keys():

                parts = act_name.split("-", 1)

                if len(parts) == 2:

                    domain = parts[0].lower()

                    # Ignore conversational acts
                    if domain not in {"general"}:
                        domains.add(domain)

            return domains

    return set()


def get_filtered_intents(dialogue, turn_id):

    # Get active intents from frames
    active_intents, active_domains = get_turn_intent_info(
        dialogue,
        turn_id
    )

    # Get domains explicitly represented in this USER utterance
    act_domains = get_current_act_domains(
        dialogue,
        turn_id
    )

    # If no task-specific dialogue act exists,
    # treat the turn as having no explicit task intent.
    if not act_domains:
        return [], set()

    # Keep only intents whose service/domain appears
    # in the current utterance's dialogue acts.
    filtered = []

    for turn in dialogue["turns"]:

        if (
            int(turn["turn_id"]) == turn_id
            and turn["speaker"] == "USER"
        ):

            for frame in turn["frames"]:

                intent = frame["state"]["active_intent"]
                service = frame["service"].lower()

                if (
                    intent != "NONE"
                    and service in act_domains
                    and intent not in filtered
                ):
                    filtered.append(intent)

            break

    return filtered, act_domains


# Test on a few transitions
test_rows = intent_transitions.sample(
    n=10,
    random_state=42
)

for _, row in test_rows.iterrows():

    dialogue = train_dialogue_dict[row["dialogue_id"]]

    prev_intents, prev_domains = get_filtered_intents(
        dialogue,
        row["previous_turn_id"]
    )

    curr_intents, curr_domains = get_filtered_intents(
        dialogue,
        row["current_turn_id"]
    )

    print("\nPrevious:", row["previous_utterance"])
    print("Filtered previous:", prev_intents, prev_domains)

    print("Current:", row["current_utterance"])
    print("Filtered current:", curr_intents, curr_domains)

    print("-" * 90)


Previous: Try 2 nights, then.
Filtered previous: ['book_hotel'] {'hotel'}
Current: Yes, I am also looking for a train that leaves after 14:15 and goes to bishops stortford.
Filtered current: ['find_train'] {'train'}
------------------------------------------------------------------------------------------

Previous: That sounds great. I would like to book that for 3 people and 2 nights starting from Sunday.
Filtered previous: ['book_hotel'] {'hotel'}
Current: No, that is all I need today. Thank you.
Filtered current: [] set()
------------------------------------------------------------------------------------------

Previous: I would like Italian please.
Filtered previous: ['find_restaurant'] {'restaurant'}
Current: It doesn't matter the part of town.
Filtered current: ['find_restaurant'] {'restaurant'}
------------------------------------------------------------------------------------------

Previous: Somewhere in the west please.
Filtered previous: ['find_hotel'] {'hotel'}
Current:

In [ ]:
# Build the final research-grade TRAIN transition dataset
# using utterance-level dialogue-act domains + filtered active intents.

def final_transition_label(
    previous_intents,
    current_intents,
    previous_domains,
    current_domains
):

    prev_intents = set(previous_intents)
    curr_intents = set(current_intents)

    prev_domains = set(previous_domains)
    curr_domains = set(current_domains)

    # No explicit current task
    if not curr_intents:
        return "NO_DRIFT"

    # Multiple current task domains/intents
    if len(curr_domains) > 1 or len(curr_intents) > 1:
        return "MULTI_GOAL"

    # No previous task
    if not prev_intents:
        return "NO_DRIFT"

    # Current goal already existed previously
    if curr_intents.issubset(prev_intents):
        return "NO_DRIFT"

    # Same domain, different task
    if prev_domains == curr_domains:
        return "GOAL_CHANGE"

    # Completely new domain
    if not curr_domains.intersection(prev_domains):
        return "DOMAIN_CHANGE"

    # Otherwise continuation
    return "NO_DRIFT"


final_rows = []

for _, row in train_transitions.iterrows():

    dialogue = train_dialogue_dict[
        row["dialogue_id"]
    ]

    # Previous turn
    prev_intents, prev_domains = get_filtered_intents(
        dialogue,
        row["previous_turn_id"]
    )

    # Current turn
    curr_intents, curr_domains = get_filtered_intents(
        dialogue,
        row["current_turn_id"]
    )

    label = final_transition_label(
        prev_intents,
        curr_intents,
        prev_domains,
        curr_domains
    )

    final_rows.append({

        "dialogue_id": row["dialogue_id"],

        "previous_turn_id": row["previous_turn_id"],
        "current_turn_id": row["current_turn_id"],

        "previous_utterance": row["previous_utterance"],
        "current_utterance": row["current_utterance"],

        "previous_intents": prev_intents,
        "current_intents": curr_intents,

        "previous_domains": sorted(prev_domains),
        "current_domains": sorted(curr_domains),

        "drift_label": label
    })


final_train_transitions = pd.DataFrame(
    final_rows
)

print("Final TRAIN transitions:",
      len(final_train_transitions))

print("\nFinal label distribution:")
print(
    final_train_transitions["drift_label"]
    .value_counts()
)

print("\nFinal percentages:")
print(
    (
        final_train_transitions["drift_label"]
        .value_counts(normalize=True) * 100
    ).round(2)
)

print("\nColumns:")
print(final_train_transitions.columns.tolist())

Final TRAIN transitions: 48339

Final label distribution:
drift_label
NO_DRIFT         35936
DOMAIN_CHANGE     5768
GOAL_CHANGE       5172
MULTI_GOAL        1463
Name: count, dtype: int64

Final percentages:
drift_label
NO_DRIFT         74.34
DOMAIN_CHANGE    11.93
GOAL_CHANGE      10.70
MULTI_GOAL        3.03
Name: proportion, dtype: float64

Columns:
['dialogue_id', 'previous_turn_id', 'current_turn_id', 'previous_utterance', 'current_utterance', 'previous_intents', 'current_intents', 'previous_domains', 'current_domains', 'drift_label']


In [ ]:
# Build USER-to-USER transitions for validation and test splits

def build_user_transitions(dialogues):

    rows = []

    for dialogue in dialogues:

        dialogue_id = dialogue["dialogue_id"]

        user_turns = [
            turn for turn in dialogue["turns"]
            if turn["speaker"] == "USER"
        ]

        user_turns = sorted(
            user_turns,
            key=lambda x: int(x["turn_id"])
        )

        for i in range(len(user_turns) - 1):

            previous = user_turns[i]
            current = user_turns[i + 1]

            rows.append({
                "dialogue_id": dialogue_id,

                "previous_turn_id": int(
                    previous["turn_id"]
                ),

                "current_turn_id": int(
                    current["turn_id"]
                ),

                "previous_utterance":
                    previous["utterance"],

                "current_utterance":
                    current["utterance"]
            })

    return pd.DataFrame(rows)


# Create transition pairs
validation_transitions = build_user_transitions(
    all_data["validation"]
)

test_transitions = build_user_transitions(
    all_data["test"]
)


print("Validation transitions:",
      len(validation_transitions))

print("Test transitions:",
      len(test_transitions))

print("\nValidation columns:")
print(validation_transitions.columns.tolist())

print("\nTest columns:")
print(test_transitions.columns.tolist())

Validation transitions: 6374
Test transitions: 6372

Validation columns:
['dialogue_id', 'previous_turn_id', 'current_turn_id', 'previous_utterance', 'current_utterance']

Test columns:
['dialogue_id', 'previous_turn_id', 'current_turn_id', 'previous_utterance', 'current_utterance']


In [ ]:
# Apply the same research-grade intent filtering and labeling
# to validation and test sets.

def label_transition_dataframe(
    transition_df,
    dialogues
):

    dialogue_dict = {
        d["dialogue_id"]: d
        for d in dialogues
    }

    labeled_rows = []

    for _, row in transition_df.iterrows():

        dialogue = dialogue_dict[row["dialogue_id"]]

        # Previous USER turn
        prev_intents, prev_domains = get_filtered_intents(
            dialogue,
            row["previous_turn_id"]
        )

        # Current USER turn
        curr_intents, curr_domains = get_filtered_intents(
            dialogue,
            row["current_turn_id"]
        )

        # Final research label
        label = final_transition_label(
            prev_intents,
            curr_intents,
            prev_domains,
            curr_domains
        )

        labeled_rows.append({

            "dialogue_id":
                row["dialogue_id"],

            "previous_turn_id":
                row["previous_turn_id"],

            "current_turn_id":
                row["current_turn_id"],

            "previous_utterance":
                row["previous_utterance"],

            "current_utterance":
                row["current_utterance"],

            "previous_intents":
                prev_intents,

            "current_intents":
                curr_intents,

            "previous_domains":
                sorted(prev_domains),

            "current_domains":
                sorted(curr_domains),

            "drift_label":
                label
        })

    return pd.DataFrame(labeled_rows)


# Validation labels
final_validation_transitions = label_transition_dataframe(
    validation_transitions,
    all_data["validation"]
)

# Test labels
final_test_transitions = label_transition_dataframe(
    test_transitions,
    all_data["test"]
)


print("VALIDATION")
print("=" * 60)
print("Total:", len(final_validation_transitions))
print(
    final_validation_transitions["drift_label"]
    .value_counts()
)
print(
    (
        final_validation_transitions["drift_label"]
        .value_counts(normalize=True) * 100
    ).round(2)
)


print("\nTEST")
print("=" * 60)
print("Total:", len(final_test_transitions))
print(
    final_test_transitions["drift_label"]
    .value_counts()
)
print(
    (
        final_test_transitions["drift_label"]
        .value_counts(normalize=True) * 100
    ).round(2)
)

VALIDATION
Total: 6374
drift_label
NO_DRIFT         4587
DOMAIN_CHANGE     849
GOAL_CHANGE       749
MULTI_GOAL        189
Name: count, dtype: int64
drift_label
NO_DRIFT         71.96
DOMAIN_CHANGE    13.32
GOAL_CHANGE      11.75
MULTI_GOAL        2.97
Name: proportion, dtype: float64

TEST
Total: 6372
drift_label
NO_DRIFT         4604
DOMAIN_CHANGE     802
GOAL_CHANGE       739
MULTI_GOAL        227
Name: count, dtype: int64
drift_label
NO_DRIFT         72.25
DOMAIN_CHANGE    12.59
GOAL_CHANGE      11.60
MULTI_GOAL        3.56
Name: proportion, dtype: float64


In [ ]:
# Prepare model-ready datasets

LABEL2ID = {
    "NO_DRIFT": 0,
    "GOAL_CHANGE": 1,
    "DOMAIN_CHANGE": 2,
    "MULTI_GOAL": 3
}

ID2LABEL = {
    v: k for k, v in LABEL2ID.items()
}


def prepare_model_dataframe(df):

    data = df[
        [
            "dialogue_id",
            "previous_turn_id",
            "current_turn_id",
            "previous_utterance",
            "current_utterance",
            "drift_label"
        ]
    ].copy()

    data["text_a"] = (
        "Previous user: "
        + data["previous_utterance"].astype(str)
    )

    data["text_b"] = (
        "Current user: "
        + data["current_utterance"].astype(str)
    )

    data["label"] = data["drift_label"].map(
        LABEL2ID
    )

    return data


train_model = prepare_model_dataframe(
    final_train_transitions
)

validation_model = prepare_model_dataframe(
    final_validation_transitions
)

test_model = prepare_model_dataframe(
    final_test_transitions
)


print("Train:", len(train_model))
print("Validation:", len(validation_model))
print("Test:", len(test_model))

print("\nLabel mapping:")
print(LABEL2ID)

print("\nExample model input:")
print("TEXT A:")
print(train_model.iloc[0]["text_a"])

print("\nTEXT B:")
print(train_model.iloc[0]["text_b"])

print("\nLABEL:")
print(
    train_model.iloc[0]["label"],
    "->",
    ID2LABEL[train_model.iloc[0]["label"]]
)

Train: 48339
Validation: 6374
Test: 6372

Label mapping:
{'NO_DRIFT': 0, 'GOAL_CHANGE': 1, 'DOMAIN_CHANGE': 2, 'MULTI_GOAL': 3}

Example model input:
TEXT A:
Previous user: I'd really like to take my client out to a nice restaurant that serves indian food.

TEXT B:
Current user: I am looking for an expensive indian restaurant in the area of centre.

LABEL:
0 -> NO_DRIFT


In [ ]:
from transformers import AutoTokenizer

# Load DeBERTa-v3 tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

print("Tokenizer loaded successfully!")
print("Tokenizer:", BASE_MODEL)
print("Vocabulary size:", tokenizer.vocab_size)

# Test sentence-pair tokenization
sample = tokenizer(
    train_model.iloc[0]["text_a"],
    train_model.iloc[0]["text_b"],
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False
)

print("\nTokenized keys:")
print(sample.keys())

print("\nNumber of tokens:",
      len(sample["input_ids"]))

print("\nFirst 20 token IDs:")
print(sample["input_ids"][:20])

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Tokenizer loaded successfully!
Tokenizer: microsoft/deberta-v3-base
Vocabulary size: 128000

Tokenized keys:
KeysView({'input_ids': [1, 6637, 1143, 294, 273, 280, 407, 431, 334, 264, 413, 312, 1648, 321, 264, 266, 1085, 2213, 272, 3442, 23991, 645, 260, 2, 6947, 1143, 294, 273, 481, 562, 270, 299, 2281, 23991, 2213, 267, 262, 537, 265, 2496, 260, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]})

Number of tokens: 42

First 20 token IDs:
[1, 6637, 1143, 294, 273, 280, 407, 431, 334, 264, 413, 312, 1648, 321, 264, 266, 1085, 2213, 272, 3442]


In [ ]:
from datasets import Dataset

# Convert pandas DataFrames to Hugging Face Datasets

train_hf = Dataset.from_pandas(
    train_model[
        ["text_a", "text_b", "label"]
    ],
    preserve_index=False
)

validation_hf = Dataset.from_pandas(
    validation_model[
        ["text_a", "text_b", "label"]
    ],
    preserve_index=False
)

test_hf = Dataset.from_pandas(
    test_model[
        ["text_a", "text_b", "label"]
    ],
    preserve_index=False
)


# Tokenization function
def tokenize_function(batch):

    return tokenizer(
        batch["text_a"],
        batch["text_b"],
        truncation=True,
        max_length=MAX_LENGTH
    )


# Tokenize datasets
tokenized_train = train_hf.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing train"
)

tokenized_validation = validation_hf.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing validation"
)

tokenized_test = test_hf.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing test"
)


print("Tokenization complete!")

print("\nTrain:", len(tokenized_train))
print("Validation:", len(tokenized_validation))
print("Test:", len(tokenized_test))

print("\nTrain columns:")
print(tokenized_train.column_names)

print("\nExample token count:",
      len(tokenized_train[0]["input_ids"]))

Tokenizing train:   0%|          | 0/48339 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/6374 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/6372 [00:00<?, ? examples/s]

Tokenization complete!

Train: 48339
Validation: 6374
Test: 6372

Train columns:
['text_a', 'text_b', 'label', 'input_ids', 'token_type_ids', 'attention_mask']

Example token count: 42


In [ ]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

# Get training labels
train_labels = np.array(
    train_model["label"].tolist()
)

# Calculate balanced class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2, 3]),
    y=train_labels
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
)

print("Class weights:")

for label_id, weight in enumerate(class_weights):
    print(
        f"{label_id} -> "
        f"{ID2LABEL[label_id]:15s} : "
        f"{weight:.4f}"
    )

print("\nTensor:")
print(class_weights)

Class weights:
0 -> NO_DRIFT        : 0.3363
1 -> GOAL_CHANGE     : 2.3366
2 -> DOMAIN_CHANGE   : 2.0951
3 -> MULTI_GOAL      : 8.2603

Tensor:
tensor([0.3363, 2.3366, 2.0951, 8.2603])


In [ ]:
from transformers import AutoModelForSequenceClassification

# Load DeBERTa-v3-base for 4-class classification

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

print("Model loaded successfully!")
print("Model:", BASE_MODEL)
print("Number of labels:", model.config.num_labels)

print("\nLabels:")
print(model.config.id2label)

# Check parameter count
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"\nTotal parameters: "
    f"{total_params:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

Model loaded successfully!
Model: microsoft/deberta-v3-base
Number of labels: 4

Labels:
{0: 'NO_DRIFT', 1: 'GOAL_CHANGE', 2: 'DOMAIN_CHANGE', 3: 'MULTI_GOAL'}

Total parameters: 184,425,220
Trainable parameters: 184,425,220


In [ ]:
from transformers import Trainer

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_fct(
            logits,
            labels
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

print("WeightedTrainer created successfully!")

WeightedTrainer created successfully!


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    _, _, f1_weighted, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro,
        "weighted_f1": f1_weighted
    }


print("Evaluation metrics defined successfully!")

Evaluation metrics defined successfully!


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./deberta_intent_drift",

    # Training
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,

    # Batch configuration
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",

    # Select best checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # Performance
    fp16=torch.cuda.is_available(),

    # Logging
    logging_steps=100,
    report_to="none",

    # Reproducibility
    seed=SEED
)

print("Training configuration created!")

print("\nEpochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Train batch size:", training_args.per_device_train_batch_size)
print(
    "Effective batch size:",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)
print("FP16:", training_args.fp16)
print("Best metric:", training_args.metric_for_best_model)

Training configuration created!

Epochs: 3
Learning rate: 2e-05
Train batch size: 8
Effective batch size: 16
FP16: True
Best metric: macro_f1


In [ ]:
# Remove original text columns that are not needed by the model

columns_to_remove = [
    "text_a",
    "text_b"
]

tokenized_train = tokenized_train.remove_columns(
    columns_to_remove
)

tokenized_validation = tokenized_validation.remove_columns(
    columns_to_remove
)

tokenized_test = tokenized_test.remove_columns(
    columns_to_remove
)

# Rename label column to the name expected by Transformers
tokenized_train = tokenized_train.rename_column(
    "label",
    "labels"
)

tokenized_validation = tokenized_validation.rename_column(
    "label",
    "labels"
)

tokenized_test = tokenized_test.rename_column(
    "label",
    "labels"
)

print("Datasets cleaned successfully!")

print("\nTrain columns:")
print(tokenized_train.column_names)

# Test collator
sample_batch = data_collator([
    tokenized_train[i]
    for i in range(2)
])

print("\nBatch shapes:")

for key, value in sample_batch.items():
    print(
        key,
        "->",
        tuple(value.shape)
    )

Datasets cleaned successfully!

Train columns:
['labels', 'input_ids', 'token_type_ids', 'attention_mask']

Batch shapes:
labels -> (2,)
input_ids -> (2, 42)
token_type_ids -> (2, 42)
attention_mask -> (2, 42)


In [ ]:
# Create the weighted Trainer using the current Transformers API

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer created successfully!")
print("Training samples:", len(tokenized_train))
print("Validation samples:", len(tokenized_validation))

Trainer created successfully!
Training samples: 48339
Validation samples: 6374


In [ ]:
# Fix FP16 gradient scaling issue
# Use full FP32 training for stability

training_args = TrainingArguments(
    output_dir="./deberta_intent_drift",
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=False,
    bf16=False,
    logging_steps=100,
    report_to="none",
    seed=SEED
)

print("Training configuration updated.")
print("FP16:", training_args.fp16)
print("BF16:", training_args.bf16)

Training configuration updated.
FP16: False
BF16: False


In [ ]:
# Recreate Trainer with the updated FP32 training configuration

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer recreated successfully!")
print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)

Trainer recreated successfully!
FP16: False
BF16: False


In [ ]:
# Start DeBERTa-v3-base training

print("Starting training...")
print("Train samples:", len(tokenized_train))
print("Validation samples:", len(tokenized_validation))
print()

train_result = trainer.train()

print("\nTraining completed successfully!")
print("Final training loss:", train_result.training_loss)

Starting training...
Train samples: 48339
Validation samples: 6374



/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,0.000000,nan,0.719642,0.179911,0.250000,0.209242,0.602317
2,0.000000,nan,0.719642,0.179911,0.250000,0.209242,0.602317
3,0.000000,nan,0.719642,0.179911,0.250000,0.209242,0.602317


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed successfully!
Final training loss: 0.06244433723010424


In [ ]:
# Check whether the trained model contains NaN or Inf parameters

import torch

nan_params = []
inf_params = []

for name, param in model.named_parameters():
    if torch.isnan(param).any():
        nan_params.append(name)
    if torch.isinf(param).any():
        inf_params.append(name)

print("NaN parameter tensors:", len(nan_params))
print("Inf parameter tensors:", len(inf_params))

if nan_params:
    print("\nFirst NaN parameters:")
    print(nan_params[:10])

if inf_params:
    print("\nFirst Inf parameters:")
    print(inf_params[:10])

if not nan_params and not inf_params:
    print("\nModel parameters are numerically valid.")

NaN parameter tensors: 202
Inf parameter tensors: 0

First NaN parameters:
['deberta.embeddings.word_embeddings.weight', 'deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.self.query_proj.weight', 'deberta.encoder.layer.0.attention.self.query_proj.bias', 'deberta.encoder.layer.0.attention.self.key_proj.weight', 'deberta.encoder.layer.0.attention.self.key_proj.bias', 'deberta.encoder.layer.0.attention.self.value_proj.weight', 'deberta.encoder.layer.0.attention.self.value_proj.bias', 'deberta.encoder.layer.0.attention.output.dense.weight']


In [ ]:
# Reload a clean pretrained DeBERTa-v3-base model
# The previous trained model is discarded because its weights contain NaNs.

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# Verify the fresh model is numerically valid
nan_count = 0
inf_count = 0

for param in model.parameters():
    nan_count += torch.isnan(param).sum().item()
    inf_count += torch.isinf(param).sum().item()

print("Fresh model loaded successfully!")
print("NaN values:", nan_count)
print("Inf values:", inf_count)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

Fresh model loaded successfully!
NaN values: 0
Inf values: 0


In [ ]:
# Stable training configuration

training_args = TrainingArguments(
    output_dir="./deberta_intent_drift",
    num_train_epochs=3,

    # More conservative learning rate
    learning_rate=1e-5,

    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # Numerical stability
    fp16=False,
    bf16=False,
    max_grad_norm=1.0,

    logging_steps=100,
    report_to="none",
    seed=SEED
)

print("Stable training configuration created.")
print("Learning rate:", training_args.learning_rate)
print("Max gradient norm:", training_args.max_grad_norm)
print("FP16:", training_args.fp16)
print("BF16:", training_args.bf16)

Stable training configuration created.
Learning rate: 1e-05
Max gradient norm: 1.0
FP16: False
BF16: False


In [ ]:
# Recreate Trainer with the stable training configuration

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Stable Trainer created successfully!")
print("Learning rate:", trainer.args.learning_rate)
print("Max gradient norm:", trainer.args.max_grad_norm)

Stable Trainer created successfully!
Learning rate: 1e-05
Max gradient norm: 1.0


In [ ]:
# Fix class-weight dtype mismatch in the custom loss

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        # Match class-weight dtype to the logits dtype
        weights = class_weights.to(
            device=logits.device,
            dtype=logits.dtype
        )

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=weights
        )

        loss = loss_fct(logits, labels)

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

print("WeightedTrainer updated successfully.")
print("Class weights:", class_weights)

WeightedTrainer updated successfully.
Class weights: tensor([0.3363, 2.3366, 2.0951, 8.2603])


In [ ]:
# Recreate Trainer with the corrected weighted loss

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer recreated successfully!")
print("Learning rate:", trainer.args.learning_rate)
print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)

Trainer recreated successfully!
Learning rate: 1e-05
FP16: False
BF16: False


In [ ]:
# Diagnose one forward + loss pass before training again

model.eval()

batch = data_collator([
    tokenized_train[i]
    for i in range(8)
])

batch = {
    key: value.to(model.device)
    for key, value in batch.items()
}

with torch.no_grad():
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"]
    )

logits = outputs.logits

weights = class_weights.to(
    device=logits.device,
    dtype=logits.dtype
)

loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
loss = loss_fct(logits, batch["labels"])

print("Logits dtype:", logits.dtype)
print("Logits min:", logits.min().item())
print("Logits max:", logits.max().item())
print("Logits contains NaN:", torch.isnan(logits).any().item())
print("Logits contains Inf:", torch.isinf(logits).any().item())
print("Loss:", loss.item())
print("Loss is NaN:", torch.isnan(loss).item())
print("Loss is Inf:", torch.isinf(loss).item())

Logits dtype: torch.float16
Logits min: nan
Logits max: nan
Logits contains NaN: True
Logits contains Inf: False
Loss: nan
Loss is NaN: True
Loss is Inf: False


In [ ]:
# Force the model to FP32 and verify the forward pass

model = model.float()

print("Model dtype after conversion:", next(model.parameters()).dtype)

model.eval()

batch = data_collator([
    tokenized_train[i]
    for i in range(8)
])

batch = {
    key: value.to(model.device)
    for key, value in batch.items()
}

with torch.no_grad():
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"]
    )

logits = outputs.logits

print("Logits dtype:", logits.dtype)
print("Logits min:", logits.min().item())
print("Logits max:", logits.max().item())
print("Logits contains NaN:", torch.isnan(logits).any().item())
print("Logits contains Inf:", torch.isinf(logits).any().item())

Model dtype after conversion: torch.float32
Logits dtype: torch.float32
Logits min: nan
Logits max: nan
Logits contains NaN: True
Logits contains Inf: False


In [ ]:
# Check the model parameters and embeddings directly

model.eval()

param_nan = []
param_inf = []

for name, param in model.named_parameters():
    if torch.isnan(param).any():
        param_nan.append(name)
    if torch.isinf(param).any():
        param_inf.append(name)

print("Parameter tensors with NaN:", len(param_nan))
print("Parameter tensors with Inf:", len(param_inf))

# Check the embedding layer directly
input_ids = batch["input_ids"]

with torch.no_grad():
    embeddings = model.deberta.embeddings(
        input_ids=input_ids
    )

print("\nEmbedding dtype:", embeddings.dtype)
print("Embedding contains NaN:", torch.isnan(embeddings).any().item())
print("Embedding contains Inf:", torch.isinf(embeddings).any().item())

print("\nFirst 10 NaN parameter names:")
print(param_nan[:10])

Parameter tensors with NaN: 202
Parameter tensors with Inf: 0

Embedding dtype: torch.float32
Embedding contains NaN: True
Embedding contains Inf: False

First 10 NaN parameter names:
['deberta.embeddings.word_embeddings.weight', 'deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.self.query_proj.weight', 'deberta.encoder.layer.0.attention.self.query_proj.bias', 'deberta.encoder.layer.0.attention.self.key_proj.weight', 'deberta.encoder.layer.0.attention.self.key_proj.bias', 'deberta.encoder.layer.0.attention.self.value_proj.weight', 'deberta.encoder.layer.0.attention.self.value_proj.bias', 'deberta.encoder.layer.0.attention.output.dense.weight']


In [ ]:
# Move the fresh model to the same device as the input batch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fresh_model = fresh_model.to(device)

print("Device:", device)
print("Model device:", next(fresh_model.parameters()).device)

with torch.no_grad():
    outputs = fresh_model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"]
    )

logits = outputs.logits

print("\nForward-pass check:")
print("Logits dtype:", logits.dtype)
print("Logits contains NaN:", torch.isnan(logits).any().item())
print("Logits contains Inf:", torch.isinf(logits).any().item())
print("Logits min:", logits.min().item())
print("Logits max:", logits.max().item())

Device: cuda
Model device: cuda:0

Forward-pass check:
Logits dtype: torch.float32
Logits contains NaN: False
Logits contains Inf: False
Logits min: -0.03795015439391136
Logits max: 0.26877251267433167


In [ ]:
# Replace the corrupted model with the verified fresh model

model = fresh_model

print("Active model replaced successfully.")
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

# Final parameter integrity check
nan_count = sum(
    torch.isnan(param).sum().item()
    for param in model.parameters()
)

inf_count = sum(
    torch.isinf(param).sum().item()
    for param in model.parameters()
)

print("NaN parameter values:", nan_count)
print("Inf parameter values:", inf_count)

Active model replaced successfully.
Model device: cuda:0
Model dtype: torch.float32
NaN parameter values: 0
Inf parameter values: 0


In [ ]:
# Recreate Trainer with the verified clean model

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer recreated with clean model.")
print("Model device:", next(trainer.model.parameters()).device)
print("Model dtype:", next(trainer.model.parameters()).dtype)
print("Learning rate:", trainer.args.learning_rate)
print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)

Trainer recreated with clean model.
Model device: cuda:0
Model dtype: torch.float32
Learning rate: 1e-05
FP16: False
BF16: False


In [ ]:
# Short stability test before full training

test_args = TrainingArguments(
    output_dir="./deberta_stability_test",
    max_steps=100,

    learning_rate=1e-5,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,

    fp16=False,
    bf16=False,
    max_grad_norm=1.0,

    logging_steps=25,
    report_to="none",
    seed=SEED
)

stability_trainer = WeightedTrainer(
    model=model,
    args=test_args,
    train_dataset=tokenized_train,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Starting 100-step stability test...")

stability_trainer.train()

# Check model parameters after 100 steps
nan_count = sum(
    torch.isnan(param).sum().item()
    for param in model.parameters()
)

inf_count = sum(
    torch.isinf(param).sum().item()
    for param in model.parameters()
)

print("\nStability test completed.")
print("NaN parameter values:", nan_count)
print("Inf parameter values:", inf_count)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Starting 100-step stability test...


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Step,Training Loss
25,2.793764
50,2.755392
75,2.688937
100,2.667833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Stability test completed.
NaN parameter values: 0
Inf parameter values: 0


In [ ]:
# Create the final Trainer from the stable 100-step model

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Final Trainer created.")
print("Training epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Training samples:", len(tokenized_train))
print("Validation samples:", len(tokenized_validation))

Final Trainer created.
Training epochs: 3
Learning rate: 1e-05
Training samples: 48339
Validation samples: 6374


In [ ]:
# Start the full 3-epoch training

print("Starting final DeBERTa training...")
print("Epochs:", training_args.num_train_epochs)
print("Train samples:", len(tokenized_train))
print("Validation samples:", len(tokenized_validation))
print()

train_result = trainer.train()

print("\nFinal training completed successfully!")
print("Final training loss:", train_result.training_loss)

Starting final DeBERTa training...
Epochs: 3
Train samples: 48339
Validation samples: 6374



/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,0.883421,0.521319,0.908692,0.776699,0.812649,0.792684,0.909906


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,0.883421,0.521319,0.908692,0.776699,0.812649,0.792684,0.909906
2,1.038591,0.503533,0.927518,0.834224,0.802561,0.810432,0.925784
3,0.851756,0.528371,0.928616,0.825060,0.816414,0.818089,0.927813


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Final training completed successfully!
Final training loss: 1.047993138008446


In [ ]:
# Verify the final trained model is numerically healthy

nan_count = 0
inf_count = 0

for param in trainer.model.parameters():
    nan_count += torch.isnan(param).sum().item()
    inf_count += torch.isinf(param).sum().item()

print("Final model verification")
print("-" * 35)
print("NaN parameter values:", nan_count)
print("Inf parameter values:", inf_count)
print("Best validation Macro F1:", trainer.state.best_metric)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

Final model verification
-----------------------------------
NaN parameter values: 0
Inf parameter values: 0
Best validation Macro F1: 0.8180892136367321
Best checkpoint: ./deberta_intent_drift/checkpoint-9066


In [ ]:
# Evaluate the best trained model on the unseen test set

test_results = trainer.evaluate(
    eval_dataset=tokenized_test,
    metric_key_prefix="test"
)

print("\nTEST SET RESULTS")
print("-" * 40)

for key, value in test_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0.851756,0.510198,3,0.922473,0.817895,0.819857,0.816008,0.921759



TEST SET RESULTS
----------------------------------------
test_loss: 0.510198
test_accuracy: 0.922473
test_macro_precision: 0.817895
test_macro_recall: 0.819857
test_macro_f1: 0.816008
test_weighted_f1: 0.921759


In [ ]:
# Generate predictions on the unseen test set
# and calculate per-class performance

from sklearn.metrics import classification_report

test_predictions = trainer.predict(tokenized_test)

test_logits = test_predictions.predictions
test_labels = test_predictions.label_ids

test_pred_ids = np.argmax(test_logits, axis=-1)

print("PER-CLASS TEST PERFORMANCE")
print("=" * 60)

print(
    classification_report(
        test_labels,
        test_pred_ids,
        labels=[0, 1, 2, 3],
        target_names=[
            "NO_DRIFT",
            "GOAL_CHANGE",
            "DOMAIN_CHANGE",
            "MULTI_GOAL"
        ],
        digits=4,
        zero_division=0
    )
)


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


PER-CLASS TEST PERFORMANCE
               precision    recall  f1-score   support

     NO_DRIFT     0.9638    0.9435    0.9536      4604
  GOAL_CHANGE     0.8380    0.9242    0.8790       739
DOMAIN_CHANGE     0.8496    0.9227    0.8846       802
   MULTI_GOAL     0.6201    0.4890    0.5468       227

     accuracy                         0.9225      6372
    macro avg     0.8179    0.8199    0.8160      6372
 weighted avg     0.9226    0.9225    0.9218      6372



In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    test_labels,
    test_pred_ids,
    labels=[0, 1, 2, 3]
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "NO_DRIFT",
        "GOAL_CHANGE",
        "DOMAIN_CHANGE",
        "MULTI_GOAL"
    ],
    columns=[
        "NO_DRIFT",
        "GOAL_CHANGE",
        "DOMAIN_CHANGE",
        "MULTI_GOAL"
    ]
)

print("TEST CONFUSION MATRIX")
print("=" * 60)
print(cm_df)

print("\nRow-wise interpretation:")
for i, label in enumerate(cm_df.index):
    print(f"\n{label}:")
    print(cm_df.iloc[i].to_dict())

TEST CONFUSION MATRIX
               NO_DRIFT  GOAL_CHANGE  DOMAIN_CHANGE  MULTI_GOAL
NO_DRIFT           4344          129             93          38
GOAL_CHANGE          53          683              2           1
DOMAIN_CHANGE        32            1            740          29
MULTI_GOAL           78            2             36         111

Row-wise interpretation:

NO_DRIFT:
{'NO_DRIFT': 4344, 'GOAL_CHANGE': 129, 'DOMAIN_CHANGE': 93, 'MULTI_GOAL': 38}

GOAL_CHANGE:
{'NO_DRIFT': 53, 'GOAL_CHANGE': 683, 'DOMAIN_CHANGE': 2, 'MULTI_GOAL': 1}

DOMAIN_CHANGE:
{'NO_DRIFT': 32, 'GOAL_CHANGE': 1, 'DOMAIN_CHANGE': 740, 'MULTI_GOAL': 29}

MULTI_GOAL:
{'NO_DRIFT': 78, 'GOAL_CHANGE': 2, 'DOMAIN_CHANGE': 36, 'MULTI_GOAL': 111}


In [ ]:
# Inspect incorrectly predicted MULTI_GOAL examples

test_analysis = test_model.copy().reset_index(drop=True)

test_analysis["true_label"] = test_labels
test_analysis["pred_label"] = test_pred_ids

test_analysis["true_class"] = test_analysis["true_label"].map(ID2LABEL)
test_analysis["pred_class"] = test_analysis["pred_label"].map(ID2LABEL)

multi_goal_errors = test_analysis[
    (test_analysis["true_class"] == "MULTI_GOAL") &
    (test_analysis["pred_class"] != "MULTI_GOAL")
].copy()

print("MULTI_GOAL errors:", len(multi_goal_errors))
print("=" * 80)

for i, (_, row) in enumerate(multi_goal_errors.head(15).iterrows(), 1):
    print(f"\nExample {i}")
    print("-" * 60)
    print("Previous:", row["previous_utterance"])
    print("Current :", row["current_utterance"])
    print("True    :", row["true_class"])
    print("Predicted:", row["pred_class"])

MULTI_GOAL errors: 116

Example 1
------------------------------------------------------------
Previous: I will also need a taxi to take me between the two place. Leaving from the restaurant at 24:30
Current : I don't think that matters. Can you just make sure I leave the restaurant to the hotel no later than 24:30.
True    : MULTI_GOAL
Predicted: NO_DRIFT

Example 2
------------------------------------------------------------
Previous: I actually don't need reservations I just need the phone number, price range.
Current : Okay. now could you help me find a restaurant in the expensive price range that is in the same area as the hotel?
True    : MULTI_GOAL
Predicted: DOMAIN_CHANGE

Example 3
------------------------------------------------------------
Previous: The restaurant should be in the east.
Current : No, I just need the post code and phone number, please.
True    : MULTI_GOAL
Predicted: NO_DRIFT

Example 4
------------------------------------------------------------
Previous: My

In [ ]:
# Analyze the structural properties of MULTI_GOAL examples

multi_goal_test = final_test_transitions[
    final_test_transitions["drift_label"] == "MULTI_GOAL"
].copy()

multi_goal_test["num_prev_intents"] = (
    multi_goal_test["previous_intents"].apply(len)
)

multi_goal_test["num_curr_intents"] = (
    multi_goal_test["current_intents"].apply(len)
)

multi_goal_test["num_prev_domains"] = (
    multi_goal_test["previous_domains"].apply(len)
)

multi_goal_test["num_curr_domains"] = (
    multi_goal_test["current_domains"].apply(len)
)

print("MULTI_GOAL STRUCTURAL ANALYSIS")
print("=" * 60)

print("\nNumber of MULTI_GOAL examples:", len(multi_goal_test))

print("\nPrevious intent counts:")
print(multi_goal_test["num_prev_intents"].value_counts().sort_index())

print("\nCurrent intent counts:")
print(multi_goal_test["num_curr_intents"].value_counts().sort_index())

print("\nPrevious domain counts:")
print(multi_goal_test["num_prev_domains"].value_counts().sort_index())

print("\nCurrent domain counts:")
print(multi_goal_test["num_curr_domains"].value_counts().sort_index())

print("\nExamples where current turn has multiple intents:")
print(
    (multi_goal_test["num_curr_intents"] > 1).sum()
)

print("\nExamples where current turn has multiple domains:")
print(
    (multi_goal_test["num_curr_domains"] > 1).sum()
)

MULTI_GOAL STRUCTURAL ANALYSIS

Number of MULTI_GOAL examples: 227

Previous intent counts:
num_prev_intents
0     12
1    193
2     20
3      2
Name: count, dtype: int64

Current intent counts:
num_curr_intents
1      5
2    211
3     11
Name: count, dtype: int64

Previous domain counts:
num_prev_domains
0     11
1    191
2     23
3      2
Name: count, dtype: int64

Current domain counts:
num_curr_domains
2    216
3     11
Name: count, dtype: int64

Examples where current turn has multiple intents:
222

Examples where current turn has multiple domains:
227


In [ ]:
# Normalized confusion matrix
# Shows the percentage distribution of predictions
# for each actual class

cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)

cm_percent = pd.DataFrame(
    cm_normalized * 100,
    index=[
        "NO_DRIFT",
        "GOAL_CHANGE",
        "DOMAIN_CHANGE",
        "MULTI_GOAL"
    ],
    columns=[
        "NO_DRIFT",
        "GOAL_CHANGE",
        "DOMAIN_CHANGE",
        "MULTI_GOAL"
    ])

print("NORMALIZED TEST CONFUSION MATRIX (%)")
print("=" * 70)

print(cm_percent.round(2))

NORMALIZED TEST CONFUSION MATRIX (%)
               NO_DRIFT  GOAL_CHANGE  DOMAIN_CHANGE  MULTI_GOAL
NO_DRIFT          94.35         2.80           2.02        0.83
GOAL_CHANGE        7.17        92.42           0.27        0.14
DOMAIN_CHANGE      3.99         0.12          92.27        3.62
MULTI_GOAL        34.36         0.88          15.86       48.90


In [ ]:
# Load SBERT for semantic similarity features

from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer(
    SBERT_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("SBERT loaded successfully")
print("Model:", SBERT_MODEL)
print("Device:", sbert_model.device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT loaded successfully
Model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda:0


In [ ]:
# Generate SBERT semantic similarity features
# for train, validation, and test transitions

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def add_sbert_similarity(df, batch_size=64):
    previous_texts = df["previous_utterance"].astype(str).tolist()
    current_texts = df["current_utterance"].astype(str).tolist()

    previous_embeddings = sbert_model.encode(
        previous_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    current_embeddings = sbert_model.encode(
        current_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarities = np.sum(
        previous_embeddings * current_embeddings,
        axis=1
    )

    result = df.copy()
    result["sbert_similarity"] = similarities

    return result


print("Generating TRAIN semantic features...")
train_enhanced = add_sbert_similarity(final_train_transitions)

print("\nGenerating VALIDATION semantic features...")
validation_enhanced = add_sbert_similarity(final_validation_transitions)

print("\nGenerating TEST semantic features...")
test_enhanced = add_sbert_similarity(final_test_transitions)

print("\nSBERT FEATURE GENERATION COMPLETE")
print("-" * 50)
print("Train:", train_enhanced.shape)
print("Validation:", validation_enhanced.shape)
print("Test:", test_enhanced.shape)

print("\nSimilarity statistics:")
print(train_enhanced["sbert_similarity"].describe())

Generating TRAIN semantic features...


Batches:   0%|          | 0/756 [00:00<?, ?it/s]

Batches:   0%|          | 0/756 [00:00<?, ?it/s]


Generating VALIDATION semantic features...


Batches:   0%|          | 0/100 [00:00<?, ?it/s]

Batches:   0%|          | 0/100 [00:00<?, ?it/s]


Generating TEST semantic features...


Batches:   0%|          | 0/100 [00:00<?, ?it/s]

Batches:   0%|          | 0/100 [00:00<?, ?it/s]


SBERT FEATURE GENERATION COMPLETE
--------------------------------------------------
Train: (48339, 11)
Validation: (6374, 11)
Test: (6372, 11)

Similarity statistics:
count    48339.000000
mean         0.271149
std          0.160845
min         -0.133218
25%          0.157689
50%          0.243332
75%          0.356726
max          1.000000
Name: sbert_similarity, dtype: float64


In [ ]:
# Create explicit intent/domain structural features

def add_structural_features(df):
    result = df.copy()

    # Convert lists safely
    result["previous_intents_set"] = result["previous_intents"].apply(set)
    result["current_intents_set"] = result["current_intents"].apply(set)

    result["previous_domains_set"] = result["previous_domains"].apply(set)
    result["current_domains_set"] = result["current_domains"].apply(set)

    # Counts
    result["previous_intent_count"] = (
        result["previous_intents_set"].apply(len)
    )

    result["current_intent_count"] = (
        result["current_intents_set"].apply(len)
    )

    result["previous_domain_count"] = (
        result["previous_domains_set"].apply(len)
    )

    result["current_domain_count"] = (
        result["current_domains_set"].apply(len)
    )

    # Overlap features
    result["intent_overlap_count"] = result.apply(
        lambda row: len(
            row["previous_intents_set"]
            & row["current_intents_set"]
        ),
        axis=1
    )

    result["domain_overlap_count"] = result.apply(
        lambda row: len(
            row["previous_domains_set"]
            & row["current_domains_set"]
        ),
        axis=1
    )

    # Binary change indicators
    result["intent_change"] = (
        result["previous_intents_set"]
        != result["current_intents_set"]
    ).astype(int)

    result["domain_change"] = (
        result["previous_domains_set"]
        != result["current_domains_set"]
    ).astype(int)

    # Multi-goal structural indicator
    result["current_multi_domain"] = (
        result["current_domain_count"] > 1
    ).astype(int)

    result["current_multi_intent"] = (
        result["current_intent_count"] > 1
    ).astype(int)

    return result


train_enhanced = add_structural_features(train_enhanced)
validation_enhanced = add_structural_features(validation_enhanced)
test_enhanced = add_structural_features(test_enhanced)

print("STRUCTURAL FEATURES CREATED")
print("-" * 50)

feature_columns = [
    "sbert_similarity",
    "previous_intent_count",
    "current_intent_count",
    "previous_domain_count",
    "current_domain_count",
    "intent_overlap_count",
    "domain_overlap_count",
    "intent_change",
    "domain_change",
    "current_multi_domain",
    "current_multi_intent"
]

print("Features:", len(feature_columns))
print(feature_columns)

print("\nTrain shape:", train_enhanced.shape)
print("Validation shape:", validation_enhanced.shape)
print("Test shape:", test_enhanced.shape)

print("\nSample feature values:")
print(
    train_enhanced[feature_columns].head()
)

STRUCTURAL FEATURES CREATED
--------------------------------------------------
Features: 11
['sbert_similarity', 'previous_intent_count', 'current_intent_count', 'previous_domain_count', 'current_domain_count', 'intent_overlap_count', 'domain_overlap_count', 'intent_change', 'domain_change', 'current_multi_domain', 'current_multi_intent']

Train shape: (48339, 25)
Validation shape: (6374, 25)
Test shape: (6372, 25)

Sample feature values:
   sbert_similarity  previous_intent_count  current_intent_count  \
0          0.692335                      1                     1   
1          0.150128                      1                     1   
2          0.294266                      1                     1   
3          0.126223                      1                     1   
4          0.055383                      1                     1   

   previous_domain_count  current_domain_count  intent_overlap_count  \
0                      1                     1                     1   
1   

In [ ]:
# Analyze how the enhanced features differ across drift classes

analysis_features = [
    "sbert_similarity",
    "previous_intent_count",
    "current_intent_count",
    "previous_domain_count",
    "current_domain_count",
    "intent_overlap_count",
    "domain_overlap_count",
    "intent_change",
    "domain_change",
    "current_multi_domain",
    "current_multi_intent"
]

feature_by_class = (
    train_enhanced
    .groupby("drift_label")[analysis_features]
    .mean()
    .round(4)
)

print("MEAN FEATURE VALUES BY DRIFT CLASS")
print("=" * 90)
print(feature_by_class)

print("\nMULTI-GOAL STRUCTURAL CHECK")
print("=" * 90)

multi_goal_stats = train_enhanced[
    train_enhanced["drift_label"] == "MULTI_GOAL"
][
    [
        "current_multi_domain",
        "current_multi_intent",
        "current_domain_count",
        "current_intent_count"
    ]
].mean()

print(multi_goal_stats.round(4))

MEAN FEATURE VALUES BY DRIFT CLASS
               sbert_similarity  previous_intent_count  current_intent_count  \
drift_label                                                                    
DOMAIN_CHANGE            0.2367                 1.0130                1.0000   
GOAL_CHANGE              0.2429                 1.0000                1.0000   
MULTI_GOAL               0.3077                 1.0369                2.0157   
NO_DRIFT                 0.2793                 0.9433                0.6999   

               previous_domain_count  current_domain_count  \
drift_label                                                  
DOMAIN_CHANGE                 1.0133                1.0000   
GOAL_CHANGE                   1.0000                1.0000   
MULTI_GOAL                    1.0403                2.0444   
NO_DRIFT                      0.9476                0.7005   

               intent_overlap_count  domain_overlap_count  intent_change  \
drift_label                        

In [ ]:
# Prepare numerical feature matrices for the enhanced feature model

from sklearn.preprocessing import StandardScaler

feature_columns = [
    "sbert_similarity",
    "previous_intent_count",
    "current_intent_count",
    "previous_domain_count",
    "current_domain_count",
    "intent_overlap_count",
    "domain_overlap_count",
    "intent_change",
    "domain_change",
    "current_multi_domain",
    "current_multi_intent"
]

X_train_features = train_enhanced[feature_columns].values
X_validation_features = validation_enhanced[feature_columns].values
X_test_features = test_enhanced[feature_columns].values

y_train_features = train_enhanced["drift_label"].map(LABEL2ID).values
y_validation_features = validation_enhanced["drift_label"].map(LABEL2ID).values
y_test_features = test_enhanced["drift_label"].map(LABEL2ID).values

# Scale only continuous/count features.
# Binary indicators remain unchanged after scaling is avoided.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_features)
X_validation_scaled = scaler.transform(X_validation_features)
X_test_scaled = scaler.transform(X_test_features)

print("FEATURE MATRICES READY")
print("-" * 50)
print("Train:", X_train_scaled.shape)
print("Validation:", X_validation_scaled.shape)
print("Test:", X_test_scaled.shape)

print("\nLabels:")
print("Train:", y_train_features.shape)
print("Validation:", y_validation_features.shape)
print("Test:", y_test_features.shape)

FEATURE MATRICES READY
--------------------------------------------------
Train: (48339, 11)
Validation: (6374, 11)
Test: (6372, 11)

Labels:
Train: (48339,)
Validation: (6374,)
Test: (6372,)


In [ ]:
# Train a feature-only baseline using Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

feature_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)

print("Training feature-only Random Forest...")

feature_model.fit(
    X_train_scaled,
    y_train_features
)

print("Training complete.")

# Validation predictions
validation_feature_pred = feature_model.predict(
    X_validation_scaled
)

# Test predictions
test_feature_pred = feature_model.predict(
    X_test_scaled
)

# Metrics
val_accuracy = accuracy_score(
    y_validation_features,
    validation_feature_pred
)

val_precision, val_recall, val_f1, _ = (
    precision_recall_fscore_support(
        y_validation_features,
        validation_feature_pred,
        average="macro",
        zero_division=0
    )
)

test_accuracy = accuracy_score(
    y_test_features,
    test_feature_pred
)

test_precision, test_recall, test_f1, _ = (
    precision_recall_fscore_support(
        y_test_features,
        test_feature_pred,
        average="macro",
        zero_division=0
    )
)

print("\nFEATURE-ONLY BASELINE")
print("=" * 50)

print("\nValidation:")
print(f"Accuracy     : {val_accuracy:.4f}")
print(f"Macro F1     : {val_f1:.4f}")
print(f"Macro Prec.  : {val_precision:.4f}")
print(f"Macro Recall : {val_recall:.4f}")

print("\nTest:")
print(f"Accuracy     : {test_accuracy:.4f}")
print(f"Macro F1     : {test_f1:.4f}")
print(f"Macro Prec.  : {test_precision:.4f}")
print(f"Macro Recall : {test_recall:.4f}")

Training feature-only Random Forest...
Training complete.

FEATURE-ONLY BASELINE

Validation:
Accuracy     : 0.9998
Macro F1     : 0.9993
Macro Prec.  : 0.9987
Macro Recall : 0.9999

Test:
Accuracy     : 1.0000
Macro F1     : 1.0000
Macro Prec.  : 1.0000
Macro Recall : 1.0000


In [ ]:
# Select structural features for the non-leaky enhanced model
# Direct label-rule indicators are intentionally excluded.

non_leaky_features = [
    "sbert_similarity",
    "previous_intent_count",
    "current_intent_count",
    "previous_domain_count",
    "current_domain_count",
    "intent_overlap_count",
    "domain_overlap_count"
]

X_train_nonleaky = train_enhanced[
    non_leaky_features
].values

X_validation_nonleaky = validation_enhanced[
    non_leaky_features
].values

X_test_nonleaky = test_enhanced[
    non_leaky_features
].values

nonleaky_scaler = StandardScaler()

X_train_nonleaky = nonleaky_scaler.fit_transform(
    X_train_nonleaky
)

X_validation_nonleaky = nonleaky_scaler.transform(
    X_validation_nonleaky
)

X_test_nonleaky = nonleaky_scaler.transform(
    X_test_nonleaky
)

print("NON-LEAKY FEATURE SET")
print("=" * 50)
print("Number of features:", len(non_leaky_features))
print(non_leaky_features)

print("\nTrain:", X_train_nonleaky.shape)
print("Validation:", X_validation_nonleaky.shape)
print("Test:", X_test_nonleaky.shape)

NON-LEAKY FEATURE SET
Number of features: 7
['sbert_similarity', 'previous_intent_count', 'current_intent_count', 'previous_domain_count', 'current_domain_count', 'intent_overlap_count', 'domain_overlap_count']

Train: (48339, 7)
Validation: (6374, 7)
Test: (6372, 7)


In [ ]:
# Train and evaluate the non-leaky feature-only baseline

nonleaky_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)

print("Training non-leaky feature-only model...")

nonleaky_model.fit(
    X_train_nonleaky,
    y_train_features
)

print("Training complete.")

# Validation
nonleaky_val_pred = nonleaky_model.predict(
    X_validation_nonleaky
)

# Test
nonleaky_test_pred = nonleaky_model.predict(
    X_test_nonleaky
)

# Metrics
val_acc = accuracy_score(
    y_validation_features,
    nonleaky_val_pred
)

_, _, val_f1, _ = precision_recall_fscore_support(
    y_validation_features,
    nonleaky_val_pred,
    average="macro",
    zero_division=0
)

test_acc = accuracy_score(
    y_test_features,
    nonleaky_test_pred
)

_, _, test_f1, _ = precision_recall_fscore_support(
    y_test_features,
    nonleaky_test_pred,
    average="macro",
    zero_division=0
)

print("\nNON-LEAKY FEATURE-ONLY RESULTS")
print("=" * 50)

print("\nValidation:")
print(f"Accuracy : {val_acc:.4f}")
print(f"Macro F1 : {val_f1:.4f}")

print("\nTest:")
print(f"Accuracy : {test_acc:.4f}")
print(f"Macro F1 : {test_f1:.4f}")

Training non-leaky feature-only model...
Training complete.

NON-LEAKY FEATURE-ONLY RESULTS

Validation:
Accuracy : 0.9998
Macro F1 : 0.9993

Test:
Accuracy : 1.0000
Macro F1 : 1.0000


In [ ]:
# Analyze SBERT semantic similarity across the four derived classes
# This uses only the actual utterance text.

sbert_by_class = (
    train_enhanced
    .groupby("drift_label")["sbert_similarity"]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)

print("SBERT SIMILARITY BY DRIFT CLASS")
print("=" * 70)
print(sbert_by_class)

print("\nTest-set SBERT similarity by class")
print("=" * 70)

test_sbert_by_class = (
    test_enhanced
    .groupby("drift_label")["sbert_similarity"]
    .agg(["mean", "std"])
    .round(4)
)

print(test_sbert_by_class)

SBERT SIMILARITY BY DRIFT CLASS
                 mean     std     min     max
drift_label                                  
DOMAIN_CHANGE  0.2367  0.1145 -0.1332  0.8627
GOAL_CHANGE    0.2429  0.1292 -0.1169  0.8739
MULTI_GOAL     0.3077  0.1632 -0.1009  0.9151
NO_DRIFT       0.2793  0.1697 -0.1214  1.0000

Test-set SBERT similarity by class
                 mean     std
drift_label                  
DOMAIN_CHANGE  0.2355  0.1052
GOAL_CHANGE    0.2541  0.1266
MULTI_GOAL     0.3200  0.1655
NO_DRIFT       0.2858  0.1653


In [ ]:
# Inspect the DeBERTa contextual representation
# that we will use for the enhanced model.

from transformers import AutoModel

encoder = AutoModel.from_pretrained(
    BASE_MODEL
)

encoder = encoder.to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

encoder.eval()

print("DeBERTa encoder loaded")
print("Device:", next(encoder.parameters()).device)

print("\nHidden size:", encoder.config.hidden_size)
print("Model type:", encoder.config.model_type)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DeBERTa encoder loaded
Device: cuda:0

Hidden size: 768
Model type: deberta-v2


In [ ]:
# Extract frozen DeBERTa representations for train/validation/test
#
# We use the BEST trained DeBERTa checkpoint already selected by Trainer.
# The encoder is frozen during feature extraction.

best_checkpoint = trainer.state.best_model_checkpoint

print("Using best checkpoint:", best_checkpoint)

best_encoder = AutoModel.from_pretrained(
    best_checkpoint
)

best_encoder = best_encoder.to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

best_encoder.eval()

device = next(best_encoder.parameters()).device

print("Encoder device:", device)
print("Hidden size:", best_encoder.config.hidden_size)

@torch.no_grad()
def extract_deberta_features(tokenized_dataset, batch_size=32):
    all_features = []

    for start in range(0, len(tokenized_dataset), batch_size):
        end = min(start + batch_size, len(tokenized_dataset))

        batch = tokenized_dataset[start:end]

        input_ids = torch.tensor(
            batch["input_ids"],
            dtype=torch.long,
            device=device
        )

        attention_mask = torch.tensor(
            batch["attention_mask"],
            dtype=torch.long,
            device=device
        )

        token_type_ids = None

        if "token_type_ids" in batch:
            token_type_ids = torch.tensor(
                batch["token_type_ids"],
                dtype=torch.long,
                device=device
            )

        outputs = best_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # First token representation
        features = outputs.last_hidden_state[:, 0, :]

        all_features.append(
            features.cpu().numpy()
        )

    return np.concatenate(all_features, axis=0)


print("\nExtracting TRAIN representations...")
X_train_deberta = extract_deberta_features(
    tokenized_train
)

print("\nExtracting VALIDATION representations...")
X_validation_deberta = extract_deberta_features(
    tokenized_validation
)

print("\nExtracting TEST representations...")
X_test_deberta = extract_deberta_features(
    tokenized_test
)

print("\nDEBERTA FEATURE EXTRACTION COMPLETE")
print("-" * 60)

print("Train:", X_train_deberta.shape)
print("Validation:", X_validation_deberta.shape)
print("Test:", X_test_deberta.shape)

Using best checkpoint: ./deberta_intent_drift/checkpoint-9066


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: ./deberta_intent_drift/checkpoint-9066
Key                 | Status     |  | 
--------------------+------------+--+-
classifier.bias     | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 
classifier.weight   | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder device: cuda:0
Hidden size: 768

Extracting TRAIN representations...


ValueError: expected sequence of length 42 at dim 1 (got 38)

In [ ]:
# Corrected DeBERTa feature extraction with dynamic padding

from torch.nn.utils.rnn import pad_sequence

@torch.no_grad()
def extract_deberta_features(tokenized_dataset, batch_size=32):
    all_features = []

    for start in range(0, len(tokenized_dataset), batch_size):
        end = min(start + batch_size, len(tokenized_dataset))

        batch = tokenized_dataset[start:end]

        # Convert each sequence separately
        input_ids_list = [
            torch.tensor(x, dtype=torch.long)
            for x in batch["input_ids"]
        ]

        attention_mask_list = [
            torch.tensor(x, dtype=torch.long)
            for x in batch["attention_mask"]
        ]

        # Dynamic padding within this batch
        input_ids = pad_sequence(
            input_ids_list,
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        ).to(device)

        attention_mask = pad_sequence(
            attention_mask_list,
            batch_first=True,
            padding_value=0
        ).to(device)

        token_type_ids = None

        if "token_type_ids" in batch:
            token_type_ids_list = [
                torch.tensor(x, dtype=torch.long)
                for x in batch["token_type_ids"]
            ]

            token_type_ids = pad_sequence(
                token_type_ids_list,
                batch_first=True,
                padding_value=0
            ).to(device)

        outputs = best_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # First-token contextual representation
        features = outputs.last_hidden_state[:, 0, :]

        all_features.append(
            features.cpu().numpy()
        )

    return np.concatenate(all_features, axis=0)


print("Extracting TRAIN representations...")
X_train_deberta = extract_deberta_features(
    tokenized_train
)

print("\nExtracting VALIDATION representations...")
X_validation_deberta = extract_deberta_features(
    tokenized_validation
)

print("\nExtracting TEST representations...")
X_test_deberta = extract_deberta_features(
    tokenized_test
)

print("\nDEBERTA FEATURE EXTRACTION COMPLETE")
print("-" * 60)

print("Train:", X_train_deberta.shape)
print("Validation:", X_validation_deberta.shape)
print("Test:", X_test_deberta.shape)

Extracting TRAIN representations...

Extracting VALIDATION representations...

Extracting TEST representations...

DEBERTA FEATURE EXTRACTION COMPLETE
------------------------------------------------------------
Train: (48339, 768)
Validation: (6374, 768)
Test: (6372, 768)


In [ ]:
# Combine DeBERTa contextual representation
# with SBERT semantic similarity

X_train_enhanced = np.concatenate(
    [
        X_train_deberta,
        train_enhanced[["sbert_similarity"]].values
    ],
    axis=1
)

X_validation_enhanced = np.concatenate(
    [
        X_validation_deberta,
        validation_enhanced[["sbert_similarity"]].values
    ],
    axis=1
)

X_test_enhanced = np.concatenate(
    [
        X_test_deberta,
        test_enhanced[["sbert_similarity"]].values
    ],
    axis=1
)

print("ENHANCED REPRESENTATIONS CREATED")
print("-" * 60)

print("DeBERTa dimensions:", X_train_deberta.shape[1])
print("SBERT similarity dimensions:", 1)
print("Combined dimensions:", X_train_enhanced.shape[1])

print("\nTrain:", X_train_enhanced.shape)
print("Validation:", X_validation_enhanced.shape)
print("Test:", X_test_enhanced.shape)

ENHANCED REPRESENTATIONS CREATED
------------------------------------------------------------
DeBERTa dimensions: 768
SBERT similarity dimensions: 1
Combined dimensions: 769

Train: (48339, 769)
Validation: (6374, 769)
Test: (6372, 769)


In [ ]:
print("Train columns:")
print(train_enhanced.columns.tolist())

print("\nValidation columns:")
print(validation_enhanced.columns.tolist())

print("\nTest columns:")
print(test_enhanced.columns.tolist())

Train columns:
['dialogue_id', 'previous_turn_id', 'current_turn_id', 'previous_utterance', 'current_utterance', 'previous_intents', 'current_intents', 'previous_domains', 'current_domains', 'drift_label', 'sbert_similarity', 'previous_intents_set', 'current_intents_set', 'previous_domains_set', 'current_domains_set', 'previous_intent_count', 'current_intent_count', 'previous_domain_count', 'current_domain_count', 'intent_overlap_count', 'domain_overlap_count', 'intent_change', 'domain_change', 'current_multi_domain', 'current_multi_intent']

Validation columns:
['dialogue_id', 'previous_turn_id', 'current_turn_id', 'previous_utterance', 'current_utterance', 'previous_intents', 'current_intents', 'previous_domains', 'current_domains', 'drift_label', 'sbert_similarity', 'previous_intents_set', 'current_intents_set', 'previous_domains_set', 'current_domains_set', 'previous_intent_count', 'current_intent_count', 'previous_domain_count', 'current_domain_count', 'intent_overlap_count', 'dom

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# ---------------------------------------------------------
# Convert enhanced features to tensors
# ---------------------------------------------------------

Xtr = torch.tensor(X_train_enhanced, dtype=torch.float32)
Xval = torch.tensor(X_validation_enhanced, dtype=torch.float32)
Xte = torch.tensor(X_test_enhanced, dtype=torch.float32)

# Correct target column
ytr = torch.tensor(
    train_enhanced["drift_label"].map(LABEL2ID).values,
    dtype=torch.long
)

yval = torch.tensor(
    validation_enhanced["drift_label"].map(LABEL2ID).values,
    dtype=torch.long
)

yte = torch.tensor(
    test_enhanced["drift_label"].map(LABEL2ID).values,
    dtype=torch.long
)

print("Feature shapes:")
print("Train:", Xtr.shape)
print("Validation:", Xval.shape)
print("Test:", Xte.shape)

print("\nLabel shapes:")
print("Train:", ytr.shape)
print("Validation:", yval.shape)
print("Test:", yte.shape)

# ---------------------------------------------------------
# DataLoaders
# ---------------------------------------------------------

train_loader = DataLoader(
    TensorDataset(Xtr, ytr),
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(Xval, yval),
    batch_size=256,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(Xte, yte),
    batch_size=256,
    shuffle=False
)

# ---------------------------------------------------------
# Enhanced MLP
# ---------------------------------------------------------

class EnhancedMLP(nn.Module):

    def __init__(self, input_dim=769, num_classes=4):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.network(x)


enhanced_model = EnhancedMLP(
    input_dim=769,
    num_classes=4
).to(device)

# ---------------------------------------------------------
# Class-weighted loss
# ---------------------------------------------------------

enhanced_class_weights = torch.tensor(
    [
        0.3363,   # NO_DRIFT
        2.3366,   # GOAL_CHANGE
        2.0951,   # DOMAIN_CHANGE
        8.2603    # MULTI_GOAL
    ],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=enhanced_class_weights
)

optimizer = torch.optim.AdamW(
    enhanced_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

num_epochs = 10

for epoch in range(num_epochs):

    enhanced_model.train()

    total_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = enhanced_model(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            enhanced_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = (
        total_loss / len(train_loader)
    )

    # -----------------------------------------------------
    # Validation
    # -----------------------------------------------------

    enhanced_model.eval()

    val_preds = []
    val_true = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)

            logits = enhanced_model(X_batch)

            preds = torch.argmax(
                logits,
                dim=1
            )

            val_preds.extend(
                preds.cpu().numpy()
            )

            val_true.extend(
                y_batch.numpy()
            )

    val_acc = accuracy_score(
        val_true,
        val_preds
    )

    val_precision, val_recall, val_f1, _ = (
        precision_recall_fscore_support(
            val_true,
            val_preds,
            average="macro",
            zero_division=0
        )
    )

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

Feature shapes:
Train: torch.Size([48339, 769])
Validation: torch.Size([6374, 769])
Test: torch.Size([6372, 769])

Label shapes:
Train: torch.Size([48339])
Validation: torch.Size([6374])
Test: torch.Size([6372])
Epoch 01/10 | Loss: 0.3883 | Val Acc: 0.8892 | Val Macro F1: 0.7692
Epoch 02/10 | Loss: 0.3429 | Val Acc: 0.8657 | Val Macro F1: 0.7528
Epoch 03/10 | Loss: 0.3388 | Val Acc: 0.8888 | Val Macro F1: 0.7696
Epoch 04/10 | Loss: 0.3347 | Val Acc: 0.8757 | Val Macro F1: 0.7585
Epoch 05/10 | Loss: 0.3327 | Val Acc: 0.8742 | Val Macro F1: 0.7596
Epoch 06/10 | Loss: 0.3321 | Val Acc: 0.8575 | Val Macro F1: 0.7434
Epoch 07/10 | Loss: 0.3326 | Val Acc: 0.8750 | Val Macro F1: 0.7586
Epoch 08/10 | Loss: 0.3278 | Val Acc: 0.8452 | Val Macro F1: 0.7350
Epoch 09/10 | Loss: 0.3290 | Val Acc: 0.8693 | Val Macro F1: 0.7565
Epoch 10/10 | Loss: 0.3285 | Val Acc: 0.8654 | Val Macro F1: 0.7506


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

# ---------------------------------------------------------
# SBERT similarity only
# ---------------------------------------------------------

X_train_sbert = train_enhanced[
    ["sbert_similarity"]
].values

X_val_sbert = validation_enhanced[
    ["sbert_similarity"]
].values

X_test_sbert = test_enhanced[
    ["sbert_similarity"]
].values

y_train_sbert = train_enhanced[
    "drift_label"
].map(LABEL2ID).values

y_val_sbert = validation_enhanced[
    "drift_label"
].map(LABEL2ID).values

y_test_sbert = test_enhanced[
    "drift_label"
].map(LABEL2ID).values

# ---------------------------------------------------------
# Simple classifier
# ---------------------------------------------------------

sbert_classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

sbert_classifier.fit(
    X_train_sbert,
    y_train_sbert
)

# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

val_pred_sbert = sbert_classifier.predict(
    X_val_sbert
)

val_acc_sbert = accuracy_score(
    y_val_sbert,
    val_pred_sbert
)

val_macro_f1_sbert = f1_score(
    y_val_sbert,
    val_pred_sbert,
    average="macro"
)

print("SBERT-ONLY VALIDATION RESULTS")
print("-" * 50)
print(f"Accuracy   : {val_acc_sbert:.4f}")
print(f"Macro F1   : {val_macro_f1_sbert:.4f}")

print("\nPer-class results:")
print(
    classification_report(
        y_val_sbert,
        val_pred_sbert,
        target_names=[
            "NO_DRIFT",
            "GOAL_CHANGE",
            "DOMAIN_CHANGE",
            "MULTI_GOAL"
        ],
        digits=4,
        zero_division=0
    )
)

SBERT-ONLY VALIDATION RESULTS
--------------------------------------------------
Accuracy   : 0.2813
Macro F1   : 0.2145

Per-class results:
               precision    recall  f1-score   support

     NO_DRIFT     0.7968    0.2470    0.3771      4587
  GOAL_CHANGE     0.1344    0.1482    0.1410       749
DOMAIN_CHANGE     0.1689    0.5830    0.2620       849
   MULTI_GOAL     0.0452    0.2857    0.0780       189

     accuracy                         0.2813      6374
    macro avg     0.2863    0.3160    0.2145      6374
 weighted avg     0.6130    0.2813    0.3251      6374



In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------

Xtr = torch.tensor(X_train_deberta, dtype=torch.float32)
Xval = torch.tensor(X_validation_deberta, dtype=torch.float32)
Xte = torch.tensor(X_test_deberta, dtype=torch.float32)

S_tr = torch.tensor(
    train_enhanced["sbert_similarity"].values,
    dtype=torch.float32
).unsqueeze(1)

S_val = torch.tensor(
    validation_enhanced["sbert_similarity"].values,
    dtype=torch.float32
).unsqueeze(1)

S_te = torch.tensor(
    test_enhanced["sbert_similarity"].values,
    dtype=torch.float32
).unsqueeze(1)

ytr = torch.tensor(
    train_enhanced["drift_label"].map(LABEL2ID).values,
    dtype=torch.long
)

yval = torch.tensor(
    validation_enhanced["drift_label"].map(LABEL2ID).values,
    dtype=torch.long
)

yte = torch.tensor(
    test_enhanced["drift_label"].map(LABEL2ID).values,
    dtype=torch.long
)

# ---------------------------------------------------------
# Dataset
# ---------------------------------------------------------

train_loader = DataLoader(
    TensorDataset(Xtr, S_tr, ytr),
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(Xval, S_val, yval),
    batch_size=256,
    shuffle=False
)

# ---------------------------------------------------------
# Gated Fusion Model
# ---------------------------------------------------------

class GatedFusionModel(nn.Module):

    def __init__(self, hidden_dim=768, num_classes=4):

        super().__init__()

        # Project SBERT similarity into the same representation space
        self.sbert_projection = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Linear(64, hidden_dim)
        )

        # Learn how much SBERT information to use
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid()
        )

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, num_classes)
        )

    def forward(self, deberta, sbert):

        # SBERT → 768-dimensional representation
        sbert_repr = self.sbert_projection(sbert)

        # Combine DeBERTa + SBERT
        combined = torch.cat(
            [deberta, sbert_repr],
            dim=1
        )

        # Learn a gate between 0 and 1
        gate = self.gate(combined)

        # Adaptive fusion
        fused = (
            gate * deberta
            +
            (1 - gate) * sbert_repr
        )

        logits = self.classifier(fused)

        return logits


fusion_model = GatedFusionModel(
    hidden_dim=768,
    num_classes=4
).to(device)

# ---------------------------------------------------------
# Class-weighted loss
# ---------------------------------------------------------

class_weights_fusion = torch.tensor(
    [
        0.3363,
        2.3366,
        2.0951,
        8.2603
    ],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_fusion
)

optimizer = torch.optim.AdamW(
    fusion_model.parameters(),
    lr=5e-5,
    weight_decay=1e-4
)

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

best_val_f1 = 0.0

for epoch in range(10):

    fusion_model.train()

    total_loss = 0.0

    for X_batch, S_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        S_batch = S_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = fusion_model(
            X_batch,
            S_batch
        )

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            fusion_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    avg_loss = (
        total_loss / len(train_loader)
    )

    # -----------------------------------------------------
    # Validation
    # -----------------------------------------------------

    fusion_model.eval()

    val_preds = []
    val_true = []

    with torch.no_grad():

        for X_batch, S_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            S_batch = S_batch.to(device)

            logits = fusion_model(
                X_batch,
                S_batch
            )

            preds = torch.argmax(
                logits,
                dim=1
            )

            val_preds.extend(
                preds.cpu().numpy()
            )

            val_true.extend(
                y_batch.numpy()
            )

    val_acc = accuracy_score(
        val_true,
        val_preds
    )

    val_macro_f1 = f1_score(
        val_true,
        val_preds,
        average="macro"
    )

    best_val_f1 = max(
        best_val_f1,
        val_macro_f1
    )

    print(
        f"Epoch {epoch+1:02d}/10 | "
        f"Loss: {avg_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

print("\nBest Validation Macro F1:", round(best_val_f1, 4))

Epoch 01/10 | Loss: 0.4121 | Val Acc: 0.8789 | Val Macro F1: 0.7614
Epoch 02/10 | Loss: 0.3429 | Val Acc: 0.8812 | Val Macro F1: 0.7642
Epoch 03/10 | Loss: 0.3360 | Val Acc: 0.8715 | Val Macro F1: 0.7543
Epoch 04/10 | Loss: 0.3351 | Val Acc: 0.8676 | Val Macro F1: 0.7551
Epoch 05/10 | Loss: 0.3314 | Val Acc: 0.8806 | Val Macro F1: 0.7632
Epoch 06/10 | Loss: 0.3305 | Val Acc: 0.8623 | Val Macro F1: 0.7491
Epoch 07/10 | Loss: 0.3280 | Val Acc: 0.8671 | Val Macro F1: 0.7514
Epoch 08/10 | Loss: 0.3255 | Val Acc: 0.8701 | Val Macro F1: 0.7546
Epoch 09/10 | Loss: 0.3257 | Val Acc: 0.8704 | Val Macro F1: 0.7553
Epoch 10/10 | Loss: 0.3256 | Val Acc: 0.8649 | Val Macro F1: 0.7511

Best Validation Macro F1: 0.7642


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
import numpy as np
import pandas as pd
import torch

# ---------------------------------------------------------
# Evaluate the BEST DeBERTa checkpoint on the test set
# ---------------------------------------------------------

test_output = trainer.predict(tokenized_test)

test_logits = test_output.predictions
test_labels = test_output.label_ids

test_predictions = np.argmax(
    test_logits,
    axis=1
)

# ---------------------------------------------------------
# Overall metrics
# ---------------------------------------------------------

test_accuracy = accuracy_score(
    test_labels,
    test_predictions
)

test_precision, test_recall, test_macro_f1, _ = (
    precision_recall_fscore_support(
        test_labels,
        test_predictions,
        average="macro",
        zero_division=0
    )
)

_, _, test_weighted_f1, _ = (
    precision_recall_fscore_support(
        test_labels,
        test_predictions,
        average="weighted",
        zero_division=0
    )
)

print("=" * 65)
print("FINAL DEBERTA TEST RESULTS")
print("=" * 65)

print(f"Accuracy          : {test_accuracy:.4f}")
print(f"Macro Precision   : {test_precision:.4f}")
print(f"Macro Recall      : {test_recall:.4f}")
print(f"Macro F1          : {test_macro_f1:.4f}")
print(f"Weighted F1       : {test_weighted_f1:.4f}")

# ---------------------------------------------------------
# Per-class results
# ---------------------------------------------------------

print("\nPER-CLASS PERFORMANCE")
print("-" * 65)

print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=[
            "NO_DRIFT",
            "GOAL_CHANGE",
            "DOMAIN_CHANGE",
            "MULTI_GOAL"
        ],
        digits=4,
        zero_division=0
    )
)

# ---------------------------------------------------------
# Confusion Matrix
# ---------------------------------------------------------

cm = confusion_matrix(
    test_labels,
    test_predictions
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "NO_DRIFT",
        "GOAL_CHANGE",
        "DOMAIN_CHANGE",
        "MULTI_GOAL"
    ],
    columns=[
        "NO_DRIFT",
        "GOAL_CHANGE",
        "DOMAIN_CHANGE",
        "MULTI_GOAL"
    ]
)

print("\nCONFUSION MATRIX")
print("-" * 65)
print(cm_df)

# ---------------------------------------------------------
# Save predictions for later analysis
# ---------------------------------------------------------

final_test_predictions = test_predictions
final_test_labels = test_labels
final_test_logits = test_logits

print("\nPredictions saved for final analysis.")
print("Number of test predictions:", len(final_test_predictions))

NameError: name 'trainer' is not defined

In [ ]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Convert logits to probabilities
# ---------------------------------------------------------

def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

test_probabilities = softmax(
    final_test_logits
)

# ---------------------------------------------------------
# Predicted class
# ---------------------------------------------------------

predicted_class_ids = np.argmax(
    test_probabilities,
    axis=1
)

predicted_drift_types = [
    ID2LABEL[int(x)]
    for x in predicted_class_ids
]

# ---------------------------------------------------------
# Intent Drift Score
# ---------------------------------------------------------

no_drift_probability = test_probabilities[:, LABEL2ID["NO_DRIFT"]]

intent_drift_scores = (
    1.0 - no_drift_probability
) * 100

# ---------------------------------------------------------
# Prediction confidence
# ---------------------------------------------------------

prediction_confidence = np.max(
    test_probabilities,
    axis=1
) * 100

# ---------------------------------------------------------
# Create final results dataframe
# ---------------------------------------------------------

drift_results = test_enhanced[
    [
        "dialogue_id",
        "previous_turn_id",
        "current_turn_id",
        "previous_utterance",
        "current_utterance",
        "drift_label"
    ]
].copy()

drift_results["predicted_drift_type"] = (
    predicted_drift_types
)

drift_results["intent_drift_score"] = (
    intent_drift_scores
)

drift_results["prediction_confidence"] = (
    prediction_confidence
)

# ---------------------------------------------------------
# Add individual class probabilities
# ---------------------------------------------------------

for class_name, class_id in LABEL2ID.items():

    drift_results[
        f"prob_{class_name.lower()}"
    ] = test_probabilities[:, class_id]

# ---------------------------------------------------------
# Display examples
# ---------------------------------------------------------

print("=" * 70)
print("INTENT DRIFT SCORE GENERATED")
print("=" * 70)

print("\nScore range:")
print(
    f"Minimum: {intent_drift_scores.min():.2f}"
)

print(
    f"Maximum: {intent_drift_scores.max():.2f}"
)

print(
    f"Mean:    {intent_drift_scores.mean():.2f}"
)

print("\nFirst 10 predictions:")
print(
    drift_results[
        [
            "previous_utterance",
            "current_utterance",
            "drift_label",
            "predicted_drift_type",
            "intent_drift_score",
            "prediction_confidence"
        ]
    ].head(10).to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# Define interpretable Drift Score bands
# ---------------------------------------------------------

def drift_level(score):
    if score < 25:
        return "LOW_DRIFT"
    elif score < 50:
        return "MODERATE_DRIFT"
    elif score < 75:
        return "HIGH_DRIFT"
    else:
        return "VERY_HIGH_DRIFT"


drift_results["drift_level"] = (
    drift_results["intent_drift_score"]
    .apply(drift_level)
)

# ---------------------------------------------------------
# Overall distribution
# ---------------------------------------------------------

print("=" * 70)
print("INTENT DRIFT SCORE ANALYSIS")
print("=" * 70)

print("\nDrift Score Statistics")
print("-" * 50)

print(
    drift_results["intent_drift_score"]
    .describe()
)

# ---------------------------------------------------------
# Drift level distribution
# ---------------------------------------------------------

print("\nDrift Level Distribution")
print("-" * 50)

level_counts = (
    drift_results["drift_level"]
    .value_counts()
)

level_percentages = (
    drift_results["drift_level"]
    .value_counts(normalize=True)
    * 100
)

for level in [
    "LOW_DRIFT",
    "MODERATE_DRIFT",
    "HIGH_DRIFT",
    "VERY_HIGH_DRIFT"
]:

    count = level_counts.get(level, 0)
    percentage = level_percentages.get(level, 0)

    print(
        f"{level:18s}: "
        f"{count:5d} "
        f"({percentage:.2f}%)"
    )

# ---------------------------------------------------------
# Drift score by actual class
# ---------------------------------------------------------

print("\nDrift Score by Actual Drift Type")
print("-" * 70)

score_by_class = (
    drift_results
    .groupby("drift_label")["intent_drift_score"]
    .agg(
        ["count", "mean", "std", "min", "median", "max"]
    )
    .reindex([
        "NO_DRIFT",
        "GOAL_CHANGE",
        "DOMAIN_CHANGE",
        "MULTI_GOAL"
    ])
)

print(
    score_by_class.round(2)
)

# ---------------------------------------------------------
# Predicted drift type distribution
# ---------------------------------------------------------

print("\nPredicted Drift Type Distribution")
print("-" * 50)

print(
    drift_results[
        "predicted_drift_type"
    ].value_counts()
)

print("\nDrift score analysis completed.")

In [ ]:
# ---------------------------------------------------------
# Prepare readable prediction examples
# ---------------------------------------------------------

examples = drift_results.copy()

examples["correct_prediction"] = (
    examples["drift_label"]
    == examples["predicted_drift_type"]
)

# ---------------------------------------------------------
# Correct predictions
# ---------------------------------------------------------

correct_examples = (
    examples[
        examples["correct_prediction"]
    ]
    .sort_values(
        "intent_drift_score",
        ascending=False
    )
)

print("=" * 80)
print("CORRECTLY CLASSIFIED HIGH-DRIFT EXAMPLES")
print("=" * 80)

for i, (_, row) in enumerate(
    correct_examples.head(5).iterrows(),
    start=1
):

    print(f"\nExample {i}")
    print("-" * 70)

    print(
        "Previous user:",
        row["previous_utterance"]
    )

    print(
        "Current user :",
        row["current_utterance"]
    )

    print(
        "Actual       :",
        row["drift_label"]
    )

    print(
        "Predicted    :",
        row["predicted_drift_type"]
    )

    print(
        "Drift Score  :",
        f"{row['intent_drift_score']:.2f}/100"
    )

    print(
        "Confidence   :",
        f"{row['prediction_confidence']:.2f}%"
    )

# ---------------------------------------------------------
# Misclassified examples
# ---------------------------------------------------------

misclassified_examples = (
    examples[
        ~examples["correct_prediction"]
    ]
    .sort_values(
        "intent_drift_score",
        ascending=False
    )
)

print("\n\n")
print("=" * 80)
print("MISCLASSIFIED EXAMPLES")
print("=" * 80)

for i, (_, row) in enumerate(
    misclassified_examples.head(5).iterrows(),
    start=1
):

    print(f"\nExample {i}")
    print("-" * 70)

    print(
        "Previous user:",
        row["previous_utterance"]
    )

    print(
        "Current user :",
        row["current_utterance"]
    )

    print(
        "Actual       :",
        row["drift_label"]
    )

    print(
        "Predicted    :",
        row["predicted_drift_type"]
    )

    print(
        "Drift Score  :",
        f"{row['intent_drift_score']:.2f}/100"
    )

    print(
        "Confidence   :",
        f"{row['prediction_confidence']:.2f}%"
    )


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# ---------------------------------------------------------
# Confusion matrix
# ---------------------------------------------------------

cm = confusion_matrix(
    final_test_labels,
    final_test_predictions
)

# Normalize row-wise
cm_normalized = (
    cm.astype(float)
    / cm.sum(axis=1, keepdims=True)
)

class_names = [
    "NO_DRIFT",
    "GOAL_CHANGE",
    "DOMAIN_CHANGE",
    "MULTI_GOAL"
]

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8, 7)
)

im = ax.imshow(
    cm_normalized,
    interpolation="nearest"
)

ax.set_title(
    "Normalized Confusion Matrix - Intent Drift Detection",
    fontsize=13
)

ax.set_xlabel(
    "Predicted Class"
)

ax.set_ylabel(
    "Actual Class"
)

ax.set_xticks(
    np.arange(len(class_names))
)

ax.set_yticks(
    np.arange(len(class_names))
)

ax.set_xticklabels(
    class_names,
    rotation=30,
    ha="right"
)

ax.set_yticklabels(
    class_names
)

# Add percentages
for i in range(
    len(class_names)
):

    for j in range(
        len(class_names)
    ):

        ax.text(
            j,
            i,
            f"{cm_normalized[i, j] * 100:.1f}%",
            ha="center",
            va="center",
            fontsize=10
        )

plt.colorbar(
    im,
    ax=ax,
    label="Proportion"
)

plt.tight_layout()

plt.show()

# ---------------------------------------------------------
# Print raw matrix
# ---------------------------------------------------------

print("\nRaw Confusion Matrix:")
print(
    pd.DataFrame(
        cm,
        index=class_names,
        columns=class_names
    )
)

In [ ]:
import pandas as pd

# ---------------------------------------------------------
# Final experimental comparison
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "Model": [
        "DeBERTa-v3-base",
        "SBERT Similarity Only",
        "DeBERTa + SBERT MLP",
        "Gated DeBERTa + SBERT"
    ],

    "Validation Accuracy": [
        0.9286,
        val_acc_sbert,
        0.8892,      # best/representative MLP result
        0.8812       # best gated-fusion accuracy
    ],

    "Validation Macro F1": [
        0.8181,
        val_macro_f1_sbert,
        0.7696,
        0.7642
    ],

    "Role": [
        "Final predictive backbone",
        "Ablation",
        "Fusion experiment",
        "Fusion experiment"
    ]
})

print("=" * 90)
print("FINAL ABLATION STUDY")
print("=" * 90)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------
# Final test performance
# ---------------------------------------------------------

print("\n")
print("=" * 90)
print("FINAL TEST PERFORMANCE — DEBERTA")
print("=" * 90)

final_test_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1"
    ],

    "Score": [
        test_accuracy,
        test_precision,
        test_recall,
        test_macro_f1,
        test_weighted_f1
    ]
})

print(
    final_test_summary.to_string(
        index=False,
        formatters={
            "Score": lambda x: f"{x:.4f}"
        }
    )
)

# ---------------------------------------------------------
# Per-class final results
# ---------------------------------------------------------

precision, recall, f1, support = (
    precision_recall_fscore_support(
        final_test_labels,
        final_test_predictions,
        labels=[0, 1, 2, 3],
        zero_division=0
    )
)

per_class_results = pd.DataFrame({
    "Class": [
        "NO_DRIFT",
        "GOAL_CHANGE",
        "DOMAIN_CHANGE",
        "MULTI_GOAL"
    ],
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Support": support
})

print("\n")
print("=" * 90)
print("FINAL PER-CLASS PERFORMANCE")
print("=" * 90)

print(
    per_class_results.to_string(
        index=False,
        formatters={
            "Precision": lambda x: f"{x:.4f}",
            "Recall": lambda x: f"{x:.4f}",
            "F1": lambda x: f"{x:.4f}"
        }
    )
)

In [ ]:
import os
import pandas as pd
import torch
import joblib
from datetime import datetime

# ============================================================
# Create output directory
# ============================================================

SAVE_DIR = "final_intent_drift_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# Save predictions
# ============================================================

prediction_columns = [
    "dialogue_id",
    "previous_utterance",
    "current_utterance",
    "drift_label",
    "predicted_drift_type",
    "prediction_confidence",
    "intent_drift_score"
]

available_columns = [
    c for c in prediction_columns
    if c in drift_results.columns
]

drift_results[available_columns].to_csv(
    os.path.join(SAVE_DIR, "test_predictions.csv"),
    index=False
)

print("Saved predictions.")

# ============================================================
# Save metrics
# ============================================================

metrics = pd.DataFrame({
    "Metric":[
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1"
    ],
    "Value":[
        test_accuracy,
        test_precision,
        test_recall,
        test_macro_f1,
        test_weighted_f1
    ]
})

metrics.to_csv(
    os.path.join(SAVE_DIR, "final_metrics.csv"),
    index=False
)

print("Saved metrics.")

# ============================================================
# Save confusion matrix
# ============================================================

cm_df = pd.DataFrame(
    cm,
    index=[
        "NO_DRIFT",
        "GOAL_CHANGE",
        "DOMAIN_CHANGE",
        "MULTI_GOAL"
    ],
    columns=[
        "NO_DRIFT",
        "GOAL_CHANGE",
        "DOMAIN_CHANGE",
        "MULTI_GOAL"
    ]
)

cm_df.to_csv(
    os.path.join(SAVE_DIR, "confusion_matrix.csv")
)

print("Saved confusion matrix.")

# ============================================================
# Save model
# ============================================================

trainer.save_model(
    os.path.join(SAVE_DIR, "deberta_model")
)

tokenizer.save_pretrained(
    os.path.join(SAVE_DIR, "deberta_model")
)

print("Saved DeBERTa model.")

# ============================================================
# Save MLP (if exists)
# ============================================================

try:
    torch.save(
        mlp_model.state_dict(),
        os.path.join(SAVE_DIR, "fusion_mlp.pt")
    )
    print("Saved fusion MLP.")
except:
    print("Fusion MLP not found (skipped).")

# ============================================================
# Save experiment summary
# ============================================================

summary = f"""
Intent Drift Detection
======================

Date:
{datetime.now()}

Dataset
-------
Train: {len(train_df)}
Validation: {len(validation_df)}
Test: {len(test_df)}

Classes
-------
NO_DRIFT
GOAL_CHANGE
DOMAIN_CHANGE
MULTI_GOAL

Final Test Accuracy:
{test_accuracy:.4f}

Final Macro F1:
{test_macro_f1:.4f}

Weighted F1:
{test_weighted_f1:.4f}

Best Checkpoint:
{trainer.state.best_model_checkpoint}
"""

with open(
    os.path.join(SAVE_DIR, "experiment_summary.txt"),
    "w"
) as f:
    f.write(summary)

print("Saved experiment summary.")

print("\n")
print("="*60)
print("PROJECT COMPLETED SUCCESSFULLY")
print("="*60)
print("Results saved to:", SAVE_DIR)